# ARC-AGI-2 offline solver

DSL program search plus a library of 33 verified programs. Runs fully offline.
Built from https://github.com/algilani2008/Kaggle-Quantum-AI (`ARC-AGI-2/`).

In [ ]:
import json, os
PIPELINE_SRC = "#!/usr/bin/env python3\n\"\"\"\nARC-AGI-2 DSL Program-Search Solver\n===================================\n\nFor each task, a library of candidate \"programs\" (DSL families) is fitted on\nthe task's train pairs. A program is accepted only if it reproduces every\ntrain output exactly (the per-task equivalent of cross-validation). Accepted\nprograms are ranked by simplicity and the top two produce attempt_1 and\nattempt_2 for each test input.\n\nDSL families\n------------\n- Geometry: identity, rotations, flips, transposes\n- Crop: crop to the non-background bounding box\n- Scaling: up-scale / down-scale by an integer factor\n- Pattern Replication:\n    * tile the input into the output with a learned transform per tile\n    * repair a periodic pattern (fill a masked colour using the detected period)\n    * repair a mirror-symmetric pattern\n- Grid Reconstruction: split the input on separator lines into cells, then\n    * combine cells pixel-wise with a learned truth table (AND/OR/XOR/...)\n    * pick the odd-one-out cell, or the cell with the most / fewest coloured pixels\n- Dynamic Shape Matching: find connected objects and\n    * extract the object selected by a property (largest, smallest, unique colour,\n      unique shape)\n    * recolour every object by a learned feature -> colour rule (size, size rank,\n      shape signature, bounding-box size)\n- Post-processing: any family can be followed by a learned colour mapping, and\n  geometry can be composed after the other families (depth-2 search)\n\nInputs\n------\n- A Kaggle-style challenges JSON ({task_id: {\"train\": [...], \"test\": [...]}}),\n  optionally with a matching solutions JSON for scoring, or\n- A directory of one-task-per-file JSONs (the public ARC-AGI-2 repo layout),\n  where test outputs are used for scoring when present.\n\nOutputs (in --out-dir)\n---------------------\n- submission.json   {task_id: [{\"attempt_1\": grid, \"attempt_2\": grid}, ...]}\n- report.json / report.md  accuracy, per-family hit counts, timing\n\nUsage\n-----\n  python pipeline.py --tasks-dir data/evaluation --out-dir output\n  python pipeline.py --challenges arc-agi_test_challenges.json --out-dir output\n  python pipeline.py --challenges ch.json --solutions sol.json --out-dir output\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom itertools import product\n\nimport numpy as np\n\nVERSION = \"1.0\"\n\n\n# --------------------------------------------------------------------------- #\n# Grid helpers\n# --------------------------------------------------------------------------- #\ndef G(x) -> np.ndarray:\n    return np.array(x, dtype=np.int16)\n\n\ndef eq(a, b) -> bool:\n    return a is not None and b is not None and a.shape == b.shape and bool((a == b).all())\n\n\ndef bg_color(g: np.ndarray) -> int:\n    vals, counts = np.unique(g, return_counts=True)\n    return int(vals[np.argmax(counts)])\n\n\ndef bbox(mask: np.ndarray):\n    rows, cols = np.where(mask)\n    if len(rows) == 0:\n        return None\n    return rows.min(), rows.max() + 1, cols.min(), cols.max() + 1\n\n\nGEOMS = {\n    \"identity\": lambda g: g,\n    \"rot90\": lambda g: np.rot90(g, 1),\n    \"rot180\": lambda g: np.rot90(g, 2),\n    \"rot270\": lambda g: np.rot90(g, 3),\n    \"flip_ud\": lambda g: g[::-1, :],\n    \"flip_lr\": lambda g: g[:, ::-1],\n    \"transpose\": lambda g: g.T,\n    \"anti_transpose\": lambda g: np.rot90(g, 2).T,\n}\n\n\ndef components(g: np.ndarray, bg: int, diag: bool, by_color: bool):\n    \"\"\"Connected components of non-background cells. Returns list of dicts.\"\"\"\n    h, w = g.shape\n    seen = np.zeros((h, w), bool)\n    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n    if diag:\n        nbrs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]\n    objs = []\n    for r in range(h):\n        for c in range(w):\n            if seen[r, c] or g[r, c] == bg:\n                continue\n            color = g[r, c]\n            stack, cells = [(r, c)], []\n            seen[r, c] = True\n            while stack:\n                y, x = stack.pop()\n                cells.append((y, x))\n                for dy, dx in nbrs:\n                    ny, nx = y + dy, x + dx\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny, nx] and g[ny, nx] != bg:\n                        if by_color and g[ny, nx] != color:\n                            continue\n                        seen[ny, nx] = True\n                        stack.append((ny, nx))\n            ys, xs = zip(*cells)\n            r0, r1, c0, c1 = min(ys), max(ys) + 1, min(xs), max(xs) + 1\n            mask = np.zeros((r1 - r0, c1 - c0), bool)\n            mask[np.array(ys) - r0, np.array(xs) - c0] = True\n            sub = g[r0:r1, c0:c1].copy()\n            sub[~mask] = bg\n            colors = Counter(int(g[y, x]) for y, x in cells)\n            objs.append(dict(cells=cells, box=(r0, r1, c0, c1), mask=mask, grid=sub,\n                             size=len(cells), color=colors.most_common(1)[0][0],\n                             shape_sig=mask.tobytes() + bytes(mask.shape)))\n    return objs\n\n\n# --------------------------------------------------------------------------- #\n# DSL families. Each: fit(pairs) -> bool, predict(grid) -> grid | None\n# --------------------------------------------------------------------------- #\nclass Family:\n    name = \"base\"\n    cost = 1\n\n    def fit(self, pairs) -> bool:\n        raise NotImplementedError\n\n    def predict(self, g):\n        raise NotImplementedError\n\n    def check(self, pairs) -> bool:\n        try:\n            return all(eq(self.predict(i), o) for i, o in pairs)\n        except Exception:\n            return False\n\n\nclass Geometry(Family):\n    cost = 1\n\n    def __init__(self, key):\n        self.key, self.name = key, f\"geom:{key}\"\n\n    def fit(self, pairs):\n        return self.check(pairs)\n\n    def predict(self, g):\n        return GEOMS[self.key](g)\n\n\nclass CropNonBg(Family):\n    name, cost = \"crop_nonbg\", 2\n\n    def fit(self, pairs):\n        return self.check(pairs)\n\n    def predict(self, g):\n        b = bbox(g != bg_color(g))\n        return None if b is None else g[b[0]:b[1], b[2]:b[3]]\n\n\nclass Scale(Family):\n    name, cost = \"scale\", 2\n\n    def fit(self, pairs):\n        i, o = pairs[0]\n        self.up = None\n        if o.shape[0] % i.shape[0] == 0 and o.shape[1] % i.shape[1] == 0:\n            self.up = (o.shape[0] // i.shape[0], o.shape[1] // i.shape[1])\n            self.name = f\"scale_up:{self.up}\"\n        elif i.shape[0] % o.shape[0] == 0 and i.shape[1] % o.shape[1] == 0:\n            self.down = (i.shape[0] // o.shape[0], i.shape[1] // o.shape[1])\n            self.name = f\"scale_down:{self.down}\"\n        else:\n            return False\n        if self.up == (1, 1):\n            return False\n        return self.check(pairs)\n\n    def predict(self, g):\n        if self.up:\n            return np.kron(g, np.ones(self.up, dtype=g.dtype))\n        return g[::self.down[0], ::self.down[1]]\n\n\n# ---- Pattern Replication --------------------------------------------------- #\nclass TileReplication(Family):\n    \"\"\"Output = (a x b) tiles of the input, each tile a learned geometric transform.\"\"\"\n    name, cost = \"pattern:tile\", 3\n\n    def fit(self, pairs):\n        i, o = pairs[0]\n        if o.shape[0] % i.shape[0] or o.shape[1] % i.shape[1]:\n            return False\n        self.a, self.b = o.shape[0] // i.shape[0], o.shape[1] // i.shape[1]\n        if (self.a, self.b) == (1, 1):\n            return False\n        h, w = i.shape\n        self.plan = {}\n        for r in range(self.a):\n            for c in range(self.b):\n                tile = o[r * h:(r + 1) * h, c * w:(c + 1) * w]\n                found = None\n                for k, f in GEOMS.items():\n                    t = f(i)\n                    if t.shape == tile.shape and (t == tile).all():\n                        found = k\n                        break\n                if found is None:\n                    if (tile == bg_color(o)).all():\n                        found = \"blank\"\n                    else:\n                        return False\n                self.plan[(r, c)] = found\n        self.name = f\"pattern:tile{self.a}x{self.b}\"\n        return self.check(pairs)\n\n    def predict(self, g):\n        h, w = g.shape\n        out = np.zeros((self.a * h, self.b * w), g.dtype)\n        for (r, c), k in self.plan.items():\n            if k == \"blank\":\n                out[r * h:(r + 1) * h, c * w:(c + 1) * w] = bg_color(g)\n            else:\n                t = GEOMS[k](g)\n                if t.shape != (h, w):\n                    return None\n                out[r * h:(r + 1) * h, c * w:(c + 1) * w] = t\n        return out\n\n\ndef _mask_color(pairs):\n    \"\"\"Colour present in every input but absent from every output (the 'hole').\"\"\"\n    cands = None\n    for i, o in pairs:\n        s = set(np.unique(i).tolist()) - set(np.unique(o).tolist())\n        cands = s if cands is None else cands & s\n    return min(cands) if cands else None\n\n\nclass PeriodicRepair(Family):\n    \"\"\"Fill a masked colour using the smallest 2-D period consistent with the rest.\"\"\"\n    name, cost = \"pattern:periodic_repair\", 3\n\n    def fit(self, pairs):\n        if any(i.shape != o.shape for i, o in pairs):\n            return False\n        self.mask = _mask_color(pairs)\n        if self.mask is None:\n            return False\n        return self.check(pairs)\n\n    @staticmethod\n    def _fill(g, known):\n        h, w = g.shape\n        for py in range(1, h + 1):\n            for px in range(1, w + 1):\n                if py == h and px == w:\n                    continue\n                tile = -np.ones((py, px), np.int16)\n                ok = True\n                for y in range(h):\n                    for x in range(w):\n                        if not known[y, x]:\n                            continue\n                        v = tile[y % py, x % px]\n                        if v == -1:\n                            tile[y % py, x % px] = g[y, x]\n                        elif v != g[y, x]:\n                            ok = False\n                            break\n                    if not ok:\n                        break\n                if ok and (tile >= 0).all():\n                    return np.tile(tile, (h // py + 1, w // px + 1))[:h, :w]\n        return None\n\n    def predict(self, g):\n        known = g != self.mask\n        if known.all():\n            return None\n        return self._fill(g, known)\n\n\nclass SymmetryRepair(Family):\n    \"\"\"Fill a masked colour from mirror / rotational copies of the grid.\"\"\"\n    name, cost = \"pattern:symmetry_repair\", 3\n\n    def fit(self, pairs):\n        if any(i.shape != o.shape for i, o in pairs):\n            return False\n        self.mask = _mask_color(pairs)\n        if self.mask is None:\n            return False\n        return self.check(pairs)\n\n    def predict(self, g):\n        out = g.copy()\n        hole = out == self.mask\n        for key in [\"flip_lr\", \"flip_ud\", \"rot180\", \"transpose\", \"anti_transpose\", \"rot90\", \"rot270\"]:\n            t = GEOMS[key](out)\n            if t.shape != out.shape:\n                continue\n            fill = hole & (t != self.mask)\n            out[fill] = t[fill]\n            hole = out == self.mask\n        return None if hole.any() else out\n\n\n# ---- Grid Reconstruction --------------------------------------------------- #\ndef split_cells(g):\n    \"\"\"Split on full-length separator rows/cols of one colour. Returns (cells, sep_color).\"\"\"\n    h, w = g.shape\n    for color in np.unique(g):\n        rows = [r for r in range(h) if (g[r] == color).all()]\n        cols = [c for c in range(w) if (g[:, c] == color).all()]\n        if not rows and not cols:\n            continue\n        rcuts = [-1] + rows + [h]\n        ccuts = [-1] + cols + [w]\n        cells = []\n        for r0, r1 in zip(rcuts, rcuts[1:]):\n            for c0, c1 in zip(ccuts, ccuts[1:]):\n                if r1 - r0 > 1 and c1 - c0 > 1:\n                    cells.append(g[r0 + 1:r1, c0 + 1:c1])\n        if len(cells) >= 2 and len({c.shape for c in cells}) == 1:\n            return cells, int(color)\n    return None, None\n\n\nclass GridCombine(Family):\n    \"\"\"Combine equal-size cells pixel-wise with a learned truth table.\"\"\"\n    name, cost = \"grid:combine_cells\", 4\n\n    def fit(self, pairs):\n        self.table = {}\n        for i, o in pairs:\n            cells, sep = split_cells(i)\n            if cells is None or cells[0].shape != o.shape:\n                return False\n            bg = self._bg(cells, sep)\n            for idx in np.ndindex(o.shape):\n                key = tuple(int(c[idx] != bg) for c in cells)\n                v = int(o[idx])\n                if self.table.setdefault(key, v) != v:\n                    return False\n        return self.check(pairs)\n\n    @staticmethod\n    def _bg(cells, sep):\n        return bg_color(np.concatenate([c.ravel() for c in cells]).reshape(1, -1))\n\n    def predict(self, g):\n        cells, sep = split_cells(g)\n        if cells is None:\n            return None\n        bg = self._bg(cells, sep)\n        out = np.zeros(cells[0].shape, np.int16)\n        for idx in np.ndindex(out.shape):\n            key = tuple(int(c[idx] != bg) for c in cells)\n            if key not in self.table:\n                return None\n            out[idx] = self.table[key]\n        return out\n\n\nclass GridSelect(Family):\n    \"\"\"Pick one cell: odd-one-out, most or fewest coloured pixels.\"\"\"\n    cost = 4\n\n    def __init__(self, rule):\n        self.rule, self.name = rule, f\"grid:select_{rule}\"\n\n    def fit(self, pairs):\n        return self.check(pairs)\n\n    def predict(self, g):\n        cells, sep = split_cells(g)\n        if cells is None:\n            return None\n        if self.rule == \"odd\":\n            keys = [c.tobytes() for c in cells]\n            cnt = Counter(keys)\n            odd = [c for c, k in zip(cells, keys) if cnt[k] == 1]\n            return odd[0] if len(odd) == 1 else None\n        bg = bg_color(np.concatenate([c.ravel() for c in cells]).reshape(1, -1))\n        counts = [int((c != bg).sum()) for c in cells]\n        target = max(counts) if self.rule == \"most\" else min(counts)\n        hits = [c for c, n in zip(cells, counts) if n == target]\n        return hits[0] if len(hits) == 1 else None\n\n\n# ---- Dynamic Shape Matching ------------------------------------------------ #\nclass ObjectSelect(Family):\n    \"\"\"Extract the object chosen by a property.\"\"\"\n    cost = 4\n\n    def __init__(self, rule, diag, by_color):\n        self.rule, self.diag, self.by_color = rule, diag, by_color\n        self.name = f\"shape:select_{rule}{'_8c' if diag else ''}{'_mc' if not by_color else ''}\"\n\n    def fit(self, pairs):\n        return self.check(pairs)\n\n    def predict(self, g):\n        objs = components(g, bg_color(g), self.diag, self.by_color)\n        if not objs:\n            return None\n        if self.rule in (\"largest\", \"smallest\"):\n            sizes = [o[\"size\"] for o in objs]\n            t = max(sizes) if self.rule == \"largest\" else min(sizes)\n            hits = [o for o in objs if o[\"size\"] == t]\n        elif self.rule == \"unique_color\":\n            cnt = Counter(o[\"color\"] for o in objs)\n            hits = [o for o in objs if cnt[o[\"color\"]] == 1]\n        else:  # unique_shape\n            cnt = Counter(o[\"shape_sig\"] for o in objs)\n            hits = [o for o in objs if cnt[o[\"shape_sig\"]] == 1]\n        return hits[0][\"grid\"] if len(hits) == 1 else None\n\n\ndef _features(objs):\n    sizes = sorted({o[\"size\"] for o in objs})\n    for o in objs:\n        yield o, {\n            \"size\": o[\"size\"],\n            \"size_rank\": sizes.index(o[\"size\"]),\n            \"size_rank_desc\": len(sizes) - 1 - sizes.index(o[\"size\"]),\n            \"shape\": o[\"shape_sig\"],\n            \"box\": o[\"mask\"].shape,\n        }\n\n\nclass ObjectRecolor(Family):\n    \"\"\"Recolour each object using a learned feature -> colour rule.\"\"\"\n    cost = 5\n\n    def __init__(self, feature, diag):\n        self.feature, self.diag = feature, diag\n        self.name = f\"shape:recolor_by_{feature}{'_8c' if diag else ''}\"\n\n    def fit(self, pairs):\n        self.rule = {}\n        for i, o in pairs:\n            if i.shape != o.shape:\n                return False\n            bg = bg_color(i)\n            if ((i == bg) != (o == bg)).any():\n                return False\n            for obj, f in _features(components(i, bg, self.diag, False)):\n                ys, xs = zip(*obj[\"cells\"])\n                colors = set(o[list(ys), list(xs)].tolist())\n                if len(colors) != 1:\n                    return False\n                c = colors.pop()\n                if self.rule.setdefault(f[self.feature], c) != c:\n                    return False\n        return len(self.rule) > 0 and self.check(pairs)\n\n    def predict(self, g):\n        bg = bg_color(g)\n        out = g.copy()\n        for obj, f in _features(components(g, bg, self.diag, False)):\n            key = f[self.feature]\n            if key not in self.rule:\n                return None\n            ys, xs = zip(*obj[\"cells\"])\n            out[list(ys), list(xs)] = self.rule[key]\n        return out\n\n\n# ---- Wrappers -------------------------------------------------------------- #\nclass WithColorMap(Family):\n    \"\"\"Run a base family, then apply a learned per-colour mapping.\"\"\"\n\n    def __init__(self, base_factory):\n        self.base_factory = base_factory\n        self.base = base_factory()\n        self.name = f\"{self.base.name}+colormap\"\n        self.cost = self.base.cost + 2\n\n    def fit(self, pairs):\n        # Let the base learn its shape parameters; its own exact-match check is\n        # expected to fail here because the colours differ.\n        try:\n            self.base.fit(pairs)\n        except Exception:\n            return False\n        mids = []\n        for i, o in pairs:\n            try:\n                m = self.base.predict(i)\n            except Exception:\n                return False\n            if m is None or m.shape != o.shape:\n                return False\n            mids.append((m, o))\n        self.cmap = {}\n        for m, o in mids:\n            for a, b in zip(m.ravel().tolist(), o.ravel().tolist()):\n                if self.cmap.setdefault(a, b) != b:\n                    return False\n        if all(k == v for k, v in self.cmap.items()):\n            return False\n        self.name = f\"{self.base.name}+colormap\"\n        return self.check(pairs)\n\n    def predict(self, g):\n        m = self.base.predict(g)\n        if m is None:\n            return None\n        out = m.copy()\n        for a, b in self.cmap.items():\n            out[m == a] = b\n        return out\n\n\nclass Compose(Family):\n    \"\"\"Geometry applied after a fitted family (depth-2 search).\"\"\"\n\n    def __init__(self, first_factory, geom_key):\n        self.first = first_factory()\n        self.geom_key = geom_key\n        self.name = f\"{self.first.name}>{geom_key}\"\n        self.cost = self.first.cost + 2\n\n    def fit(self, pairs):\n        inv = {\"rot90\": \"rot270\", \"rot270\": \"rot90\"}.get(self.geom_key, self.geom_key)\n        back = [(i, GEOMS[inv](o)) for i, o in pairs]\n        if not self.first.fit(back):\n            return False\n        self.name = f\"{self.first.name}>{self.geom_key}\"\n        return self.check(pairs)\n\n    def predict(self, g):\n        m = self.first.predict(g)\n        return None if m is None else GEOMS[self.geom_key](m)\n\n\n# ---- Program library (verified programs from earlier searches) ------------ #\nLIB_RUNNER = r'''\nimport json, sys, signal, resource\nresource.setrlimit(resource.RLIMIT_AS, (2 * 1024 ** 3, 2 * 1024 ** 3))\ndef _alarm(*a):\n    raise TimeoutError(\"program timeout\")\nsignal.signal(signal.SIGALRM, _alarm)\nreq = json.load(sys.stdin)\nres = {}\nfor name, src in req[\"programs\"].items():\n    ns = {\"__name__\": \"lib_\" + name}\n    outs = []\n    try:\n        signal.alarm(2)\n        exec(compile(src, name, \"exec\"), ns)\n        fn = ns[\"transform\"]\n        for g in req[\"grids\"]:\n            r = fn([row[:] for row in g])\n            if hasattr(r, \"tolist\"):\n                r = r.tolist()\n            outs.append([[int(v) for v in row] for row in r])\n        signal.alarm(0)\n    except BaseException:\n        signal.alarm(0)\n        outs = None\n    res[name] = outs\nprint(json.dumps(res))\n'''\n\n_LIB_CACHE = {}\n\n\ndef load_program_library(path):\n    \"\"\"Load every *.py in `path` that defines transform(). Cached per path.\"\"\"\n    if path not in _LIB_CACHE:\n        progs = {}\n        if path and os.path.isdir(path):\n            for fn in sorted(os.listdir(path)):\n                if fn.endswith(\".py\"):\n                    with open(os.path.join(path, fn)) as f:\n                        src = f.read()\n                    if \"def transform\" in src:\n                        progs[fn[:-3]] = src\n        _LIB_CACHE[path] = progs\n    return _LIB_CACHE[path]\n\n\ndef run_library(progs, grids, timeout):\n    \"\"\"Run every library program on every grid in one isolated subprocess.\"\"\"\n    import subprocess\n    if not progs:\n        return {}\n    try:\n        p = subprocess.run([sys.executable, \"-I\", \"-c\", LIB_RUNNER],\n                           input=json.dumps({\"programs\": progs, \"grids\": grids}),\n                           capture_output=True, text=True, timeout=timeout,\n                           env={\"PATH\": os.environ.get(\"PATH\", \"\")})\n        return json.loads(p.stdout)\n    except Exception:\n        return {}\n\n\ndef library_candidates(task, lib_path, timeout=60):\n    \"\"\"Library programs that reproduce every train pair, with their test predictions.\"\"\"\n    progs = load_program_library(lib_path)\n    if not progs:\n        return []\n    train_in = [p[\"input\"] for p in task[\"train\"]]\n    train_out = [p[\"output\"] for p in task[\"train\"]]\n    test_in = [t[\"input\"] for t in task[\"test\"]]\n    res = run_library(progs, train_in + test_in, timeout)\n    hits = []\n    for name, outs in res.items():\n        if outs and outs[:len(train_in)] == train_out:\n            preds = outs[len(train_in):]\n            if all(0 < len(g) <= 30 and all(0 < len(r) <= 30 for r in g) for g in preds):\n                hits.append((f\"library:{name}\", preds))\n    return hits\n\n\ndef library():\n    \"\"\"Factories for every candidate program, cheapest first.\"\"\"\n    fs = [lambda k=k: Geometry(k) for k in GEOMS]\n    fs += [CropNonBg, Scale, TileReplication, PeriodicRepair, SymmetryRepair, GridCombine]\n    fs += [lambda r=r: GridSelect(r) for r in (\"odd\", \"most\", \"fewest\")]\n    for rule, diag, by_color in product((\"largest\", \"smallest\", \"unique_color\", \"unique_shape\"),\n                                        (False, True), (True, False)):\n        fs.append(lambda r=rule, d=diag, b=by_color: ObjectSelect(r, d, b))\n    for feat, diag in product((\"size\", \"size_rank\", \"size_rank_desc\", \"shape\", \"box\"), (False, True)):\n        fs.append(lambda f=feat, d=diag: ObjectRecolor(f, d))\n    base_for_cmap = [lambda k=k: Geometry(k) for k in GEOMS] + [CropNonBg, Scale]\n    base_for_cmap += [lambda r=r: GridSelect(r) for r in (\"odd\", \"most\", \"fewest\")]\n    fs += [lambda b=b: WithColorMap(b) for b in base_for_cmap]\n    composable = [CropNonBg, TileReplication, GridCombine] + \\\n                 [lambda r=r: ObjectSelect(r, False, True) for r in (\"largest\", \"smallest\", \"unique_color\", \"unique_shape\")]\n    fs += [lambda f=f, k=k: Compose(f, k) for f in composable for k in GEOMS if k != \"identity\"]\n    return fs\n\n\n# --------------------------------------------------------------------------- #\n# Solver\n# --------------------------------------------------------------------------- #\ndef solve_task(task, time_budget=10.0, lib_path=None):\n    \"\"\"Return (list of attempts per test input, list of fitted family names).\"\"\"\n    lib_hits = library_candidates(task, lib_path) if lib_path else []\n    pairs = [(G(p[\"input\"]), G(p[\"output\"])) for p in task[\"train\"]]\n    t0 = time.time()\n    fitted = []\n    for factory in library():\n        if time.time() - t0 > time_budget:\n            break\n        fam = factory()\n        try:\n            if fam.fit(pairs):\n                fitted.append(fam)\n        except Exception:\n            continue\n    fitted.sort(key=lambda f: f.cost)\n\n    results = []\n    for ti, t in enumerate(task[\"test\"]):\n        g = G(t[\"input\"])\n        attempts, seen = [], set()\n        for name, preds in lib_hits:\n            p = G(preds[ti])\n            key = p.tobytes() + bytes(p.shape)\n            if key not in seen and len(attempts) < 2:\n                seen.add(key)\n                attempts.append((p.tolist(), name))\n        for fam in fitted:\n            if len(attempts) == 2:\n                break\n            try:\n                p = fam.predict(g)\n            except Exception:\n                p = None\n            if p is None or p.size == 0 or p.shape[0] > 30 or p.shape[1] > 30:\n                continue\n            key = p.tobytes() + bytes(p.shape)\n            if key in seen:\n                continue\n            seen.add(key)\n            attempts.append((p.tolist(), fam.name))\n            if len(attempts) == 2:\n                break\n        while len(attempts) < 2:\n            attempts.append((g.tolist(), \"fallback:input\"))\n        results.append(attempts)\n    return results, [n for n, _ in lib_hits] + [f.name for f in fitted]\n\n\n# --------------------------------------------------------------------------- #\n# I/O and scoring\n# --------------------------------------------------------------------------- #\ndef load_tasks(args):\n    if args.challenges:\n        with open(args.challenges) as f:\n            tasks = json.load(f)\n        sols = None\n        if args.solutions:\n            with open(args.solutions) as f:\n                sols = json.load(f)\n        return tasks, sols\n    tasks, sols = {}, {}\n    for fn in sorted(os.listdir(args.tasks_dir)):\n        if not fn.endswith(\".json\"):\n            continue\n        with open(os.path.join(args.tasks_dir, fn)) as f:\n            t = json.load(f)\n        tid = fn[:-5]\n        tasks[tid] = {\"train\": t[\"train\"], \"test\": [{\"input\": x[\"input\"]} for x in t[\"test\"]]}\n        if all(\"output\" in x for x in t[\"test\"]):\n            sols[tid] = [x[\"output\"] for x in t[\"test\"]]\n    return tasks, (sols or None)\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    src = ap.add_mutually_exclusive_group(required=True)\n    src.add_argument(\"--tasks-dir\", help=\"directory of one-task-per-file JSONs\")\n    src.add_argument(\"--challenges\", help=\"Kaggle-style challenges JSON\")\n    ap.add_argument(\"--solutions\", help=\"Kaggle-style solutions JSON (for scoring)\")\n    ap.add_argument(\"--out-dir\", default=\"output\")\n    ap.add_argument(\"--time-per-task\", type=float, default=10.0)\n    ap.add_argument(\"--limit\", type=int, default=0, help=\"only the first N tasks (0 = all)\")\n    ap.add_argument(\"--program-library\",\n                    default=os.path.join(os.path.dirname(os.path.abspath(__file__)), \"gemini\", \"results\", \"programs\"),\n                    help=\"folder of verified transform() programs tried on every task ('' to disable)\")\n    args = ap.parse_args()\n\n    t0 = time.time()\n    os.makedirs(args.out_dir, exist_ok=True)\n    tasks, sols = load_tasks(args)\n    ids = list(tasks)[: args.limit or None]\n\n    submission, per_task, fam_hits, fam_fits = {}, {}, Counter(), Counter()\n    solved = total = 0\n    for n, tid in enumerate(ids, 1):\n        results, fitted = solve_task(tasks[tid], args.time_per_task, args.program_library or None)\n        fam_fits.update(set(fitted))\n        submission[tid] = [{\"attempt_1\": a[0][0], \"attempt_2\": a[1][0]} for a in results]\n        rec = {\"fitted\": fitted, \"attempts\": [[a[0][1], a[1][1]] for a in results]}\n        if sols and tid in sols:\n            ok = []\n            for a, truth in zip(results, sols[tid]):\n                hit = next((name for grid, name in a if grid == truth), None)\n                ok.append(hit is not None)\n                if hit:\n                    fam_hits[hit] += 1\n            # ARC scoring: each test input of a task counts as a fraction of the task\n            solved += sum(ok) / len(ok)\n            total += 1\n            rec[\"correct\"] = ok\n        per_task[tid] = rec\n        if n % 50 == 0:\n            print(f\"[{time.time() - t0:6.1f}s] {n}/{len(ids)} tasks\", flush=True)\n\n    with open(os.path.join(args.out_dir, \"submission.json\"), \"w\") as f:\n        json.dump(submission, f)\n\n    acc = solved / total if total else None\n    any_fit = sum(1 for r in per_task.values() if r[\"fitted\"])\n    report = dict(\n        version=VERSION, run_at=datetime.now(timezone.utc).isoformat(),\n        source=args.challenges or args.tasks_dir, n_tasks=len(ids),\n        tasks_with_a_fitting_program=any_fit,\n        scored=bool(total), pass_at_2=acc, tasks_solved=round(solved, 3),\n        correct_by_family=dict(fam_hits.most_common()),\n        fitted_by_family=dict(fam_fits.most_common()),\n        library_size=len(library()), runtime_sec=round(time.time() - t0, 1),\n        per_task=per_task,\n    )\n    with open(os.path.join(args.out_dir, \"report.json\"), \"w\") as f:\n        json.dump(report, f, indent=1)\n\n    md = [f\"# ARC-AGI-2 solver report \u2014 {report['run_at']}\", \"\",\n          f\"- Source: `{report['source']}`\",\n          f\"- Tasks: {len(ids)}; DSL programs in library: {report['library_size']}\",\n          f\"- Tasks where at least one program fit all train pairs: {any_fit} ({any_fit / max(len(ids), 1):.1%})\"]\n    if total:\n        md.append(f\"- **pass@2 accuracy: {acc:.2%}** ({solved:.2f} / {total} tasks)\")\n        md += [\"\", \"## Correct predictions by DSL family\", \"\"] + \\\n              [f\"- {k}: {v}\" for k, v in fam_hits.most_common()] or [\"- none\"]\n    else:\n        md.append(\"- No solutions provided, so accuracy was not scored.\")\n    md += [\"\", \"## Programs that fit train pairs (top 15)\", \"\"] + \\\n          [f\"- {k}: {v}\" for k, v in fam_fits.most_common(15)]\n    md.append(f\"\\nRuntime: {report['runtime_sec']}s\")\n    with open(os.path.join(args.out_dir, \"report.md\"), \"w\") as f:\n        f.write(\"\\n\".join(md) + \"\\n\")\n    print(\"\\n\".join(md))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
PROGRAMS = {"0934a4d8.py": "# ARC-AGI-2 task 0934a4d8: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    H, W = arr.shape\n\n    # Find the vertical symmetry sum R_sum (r + r' = R_sum)\n    best_r_sum = H - 1\n    best_r_score = (-1.0, -1)\n    for r_sum in range(H // 2, 2 * H - H // 2):\n        matches = 0\n        total = 0\n        for r in range(H):\n            r2 = r_sum - r\n            if 0 <= r2 < H:\n                mask = (arr[r, :] != 8) & (arr[r2, :] != 8)\n                total += int(np.sum(mask))\n                matches += int(np.sum(arr[r, mask] == arr[r2, mask]))\n        if total > 0:\n            score = (matches / total, matches)\n            if score > best_r_score:\n                best_r_score = score\n                best_r_sum = r_sum\n\n    # Find the horizontal symmetry sum C_sum (c + c' = C_sum)\n    best_c_sum = W - 1\n    best_c_score = (-1.0, -1)\n    for c_sum in range(W // 2, 2 * W - W // 2):\n        matches = 0\n        total = 0\n        for c in range(W):\n            c2 = c_sum - c\n            if 0 <= c2 < W:\n                mask = (arr[:, c] != 8) & (arr[:, c2] != 8)\n                total += int(np.sum(mask))\n                matches += int(np.sum(arr[mask, c] == arr[mask, c2]))\n        if total > 0:\n            score = (matches / total, matches)\n            if score > best_c_score:\n                best_c_score = score\n                best_c_sum = c_sum\n\n    # Locate the rectangular region of 8s\n    rows, cols = np.where(arr == 8)\n    r_min, r_max = int(rows.min()), int(rows.max())\n    c_min, c_max = int(cols.min()), int(cols.max())\n\n    out = []\n    for r in range(r_min, r_max + 1):\n        row_vals = []\n        for c in range(c_min, c_max + 1):\n            val = arr[r, c]\n            if val == 8:\n                candidates = [\n                    (best_r_sum - r, c),\n                    (r, best_c_sum - c),\n                    (best_r_sum - r, best_c_sum - c),\n                    (c, r),\n                    (c, best_r_sum - r),\n                    (best_c_sum - c, r),\n                    (best_c_sum - c, best_r_sum - r),\n                ]\n                for nr, nc in candidates:\n                    if 0 <= nr < H and 0 <= nc < W and arr[nr, nc] != 8:\n                        val = arr[nr, nc]\n                        break\n            row_vals.append(int(val))\n        out.append(row_vals)\n\n    return out\n", "135a2760.py": "# ARC-AGI-2 task 135a2760: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n    bg = arr[0, 0]\n    out = arr.copy()\n\n    # Find all minimal hollow rectangular frames of uniform non-background colour\n    frames = []\n    for r1 in range(H - 2):\n        for c1 in range(W - 2):\n            fc = arr[r1, c1]\n            if fc == bg:\n                continue\n            for r2 in range(r1 + 2, H):\n                if arr[r2, c1] != fc:\n                    break\n                for c2 in range(c1 + 2, W):\n                    if arr[r1, c2] != fc:\n                        break\n                    if (\n                        np.all(arr[r1, c1 : c2 + 1] == fc)\n                        and np.all(arr[r2, c1 : c2 + 1] == fc)\n                        and np.all(arr[r1 : r2 + 1, c1] == fc)\n                        and np.all(arr[r1 : r2 + 1, c2] == fc)\n                    ):\n                        # Ensure no interior row or column is completely frame-coloured\n                        sub = arr[r1 + 1 : r2, c1 + 1 : c2]\n                        if not np.any(np.all(sub == fc, axis=1)) and not np.any(\n                            np.all(sub == fc, axis=0)\n                        ):\n                            frames.append((r1 + 1, c1 + 1, r2 - 1, c2 - 1))\n\n    for ir1, ic1, ir2, ic2 in frames:\n        sub = arr[ir1 : ir2 + 1, ic1 : ic2 + 1]\n        sh, sw = sub.shape\n        transposed = sw > sh\n        if transposed:\n            sub = sub.T  # shape: (L, D) where L is the long axis\n        L, D = sub.shape\n\n        best_score = float(\"inf\")\n        best_recon = sub\n\n        max_p = max(1, L // 3)\n        for p in range(1, max_p + 1):\n            recon = np.empty_like(sub)\n            mismatches = 0\n            for rem in range(p):\n                indices = range(rem, L, p)\n                for d in range(D):\n                    vals = [sub[i, d] for i in indices]\n                    mode = Counter(vals).most_common(1)[0][0]\n                    for i in indices:\n                        recon[i, d] = mode\n                        if sub[i, d] != mode:\n                            mismatches += 1\n            score = mismatches + 0.4 * p\n            if score < best_score:\n                best_score = score\n                best_recon = recon\n\n        if transposed:\n            best_recon = best_recon.T\n        out[ir1 : ir2 + 1, ic1 : ic2 + 1] = best_recon\n\n    return out.tolist()\n", "136b0064.py": "# ARC-AGI-2 task 136b0064: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    sep_col = next(c for c in range(len(grid[0])) if all(grid[r][c] == 4 for r in range(H)))\n    \n    out = [row[sep_col + 1:] for row in grid]\n    c = out[0].index(5)\n    r = 1\n    \n    for bc in range(0, sep_col, 4):\n        for br in range(0, H, 4):\n            block = [grid[br + dr][bc : bc + 3] for dr in range(3)]\n            color = next((val for row in block for val in row if val != 0), 0)\n            \n            if color == 1:\n                for dc in range(3):\n                    out[r][c + dc] = 1\n                c += 2\n                r += 1\n            elif color == 2:\n                for dc in range(2):\n                    out[r][c - dc] = 2\n                c -= 1\n                r += 1\n            elif color == 3:\n                for dc in range(4):\n                    out[r][c - dc] = 3\n                c -= 3\n                r += 1\n            elif color == 6:\n                out[r][c] = 6\n                out[r + 1][c] = 6\n                r += 2\n                \n    return out\n", "13e47133.py": "# ARC-AGI-2 task 13e47133: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter, deque\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H, W = len(grid), len(grid[0])\n    counts = Counter(val for row in grid for val in row)\n    (bg, _), (wall, _) = counts.most_common(2)\n\n    out = [row[:] for row in grid]\n    visited = [[False] * W for _ in range(H)]\n\n    for r0 in range(H):\n        for c0 in range(W):\n            if grid[r0][c0] == wall or visited[r0][c0]:\n                continue\n\n            # Find 4-connected region of non-wall cells\n            comp = []\n            q = deque([(r0, c0)])\n            visited[r0][c0] = True\n            while q:\n                r, c = q.popleft()\n                comp.append((r, c))\n                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                    nr, nc = r + dr, c + dc\n                    if 0 <= nr < H and 0 <= nc < W and not visited[nr][nc] and grid[nr][nc] != wall:\n                        visited[nr][nc] = True\n                        q.append((nr, nc))\n\n            comp_set = set(comp)\n            outside = [\n                (r, c)\n                for r in range(H)\n                for c in range(W)\n                if (r, c) not in comp_set\n            ]\n\n            # Compute 0-indexed Chebyshev layer for each cell in the region\n            layers = {}\n            clues = {}\n            for r, c in comp:\n                dist = min(r + 1, H - r, c + 1, W - c)\n                for or_, oc in outside:\n                    d = max(abs(r - or_), abs(c - oc))\n                    if d < dist:\n                        dist = d\n                layer = dist - 1\n                layers[(r, c)] = layer\n                if grid[r][c] != bg:\n                    clues[layer] = grid[r][c]\n\n            if not clues:\n                continue\n\n            length = max(clues) + 1\n            seq = [clues.get(i, bg) for i in range(length)]\n\n            for r, c in comp:\n                out[r][c] = seq[layers[(r, c)] % length]\n\n    return out\n", "142ca369.py": "# ARC-AGI-2 task 142ca369: verified on all train pairs\n# model=gemini-4-argon sample=1 round=2 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    H, W = arr.shape\n    output = arr.copy()\n\n    visited = np.zeros((H, W), dtype=bool)\n    emitters = []\n    reflectors = {}\n\n    # Find monochromatic 4-connected components\n    for r in range(H):\n        for c in range(W):\n            if arr[r, c] != 0 and not visited[r, c]:\n                color = int(arr[r, c])\n                comp = []\n                queue = [(r, c)]\n                visited[r, c] = True\n                while queue:\n                    cr, cc = queue.pop(0)\n                    comp.append((cr, cc))\n                    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and not visited[nr, nc] and arr[nr, nc] == color:\n                            visited[nr, nc] = True\n                            queue.append((nr, nc))\n\n                rows = [p[0] for p in comp]\n                cols = [p[1] for p in comp]\n                # L-shape emitter: 3 cells in a 2x2 bounding box\n                if len(comp) == 3 and (max(rows) - min(rows) == 1) and (max(cols) - min(cols) == 1):\n                    comp_set = set(comp)\n                    for pr, pc in comp:\n                        neighbors = [\n                            (nr, nc)\n                            for nr, nc in [(pr - 1, pc), (pr + 1, pc), (pr, pc - 1), (pr, pc + 1)]\n                            if (nr, nc) in comp_set\n                        ]\n                        if len(neighbors) == 2:\n                            dr = 2 * pr - neighbors[0][0] - neighbors[1][0]\n                            dc = 2 * pc - neighbors[0][1] - neighbors[1][1]\n                            emitters.append((pr, pc, dr, dc, color))\n                            break\n                else:\n                    # Reflector (1x1 dot, 1x3 bar, or 3x1 bar): record its center cell\n                    center_r = sum(rows) // len(comp)\n                    center_c = sum(cols) // len(comp)\n                    reflectors[(center_r, center_c)] = color\n\n    # Trace rays from each emitter\n    for vr, vc, dr, dc, color in emitters:\n        r, c = vr + dr, vc + dc\n        cur_color = color\n        while 0 <= r < H and 0 <= c < W:\n            if (r + dr, c) in reflectors:\n                cur_color = reflectors[(r + dr, c)]\n                dr = -dr\n            elif (r, c + dc) in reflectors:\n                cur_color = reflectors[(r, c + dc)]\n                dc = -dc\n            output[r, c] = cur_color\n            r += dr\n            c += dc\n\n    return output.tolist()\n", "16b78196.py": "# ARC-AGI-2 task 16b78196: verified on all train pairs\n# model=gemini-4-argon sample=1 round=1 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n\n    # Find monochromatic 8-connected components\n    visited = np.zeros((H, W), dtype=bool)\n    components = []\n    for r in range(H):\n        for c in range(W):\n            if arr[r, c] != 0 and not visited[r, c]:\n                color = arr[r, c]\n                comp = []\n                queue = [(r, c)]\n                visited[r, c] = True\n                while queue:\n                    cr, cc = queue.pop()\n                    comp.append((cr, cc))\n                    for dr in (-1, 0, 1):\n                        for dc in (-1, 0, 1):\n                            nr, nc = cr + dr, cc + dc\n                            if 0 <= nr < H and 0 <= nc < W:\n                                if not visited[nr, nc] and arr[nr, nc] == color:\n                                    visited[nr, nc] = True\n                                    queue.append((nr, nc))\n                components.append((color, comp))\n\n    # Identify the wall component (spans across the grid horizontally or vertically)\n    wall_idx = -1\n    is_vertical = False\n    for i, (color, comp) in enumerate(components):\n        rs = [p[0] for p in comp]\n        cs = [p[1] for p in comp]\n        if min(cs) == 0 and max(cs) == W - 1:\n            wall_idx = i\n            is_vertical = False\n            break\n        if min(rs) == 0 and max(rs) == H - 1:\n            wall_idx = i\n            is_vertical = True\n            break\n\n    if is_vertical:\n        return np.array(transform(arr.T.tolist())).T.tolist()\n\n    # Build base grid with only the horizontal wall\n    base = np.zeros((H, W), dtype=int)\n    wall_color, wall_comp = components[wall_idx]\n    for r, c in wall_comp:\n        base[r, c] = wall_color\n\n    # Extract movable pieces as cropped 2D arrays\n    pieces = []\n    for i, (color, comp) in enumerate(components):\n        if i == wall_idx:\n            continue\n        rs = [p[0] for p in comp]\n        cs = [p[1] for p in comp]\n        r0, r1 = min(rs), max(rs)\n        c0, c1 = min(cs), max(cs)\n        patch = np.zeros((r1 - r0 + 1, c1 - c0 + 1), dtype=int)\n        for r, c in comp:\n            patch[r - r0, c - c0] = color\n        pieces.append(patch)\n\n    num_pieces = len(pieces)\n    valid_stacks = []\n\n    # Search for valid stacks on both 'bottom' and 'top' (by vertically flipping 'top')\n    for side in (\"bottom\", \"top\"):\n        work_base = base if side == \"bottom\" else base[::-1, :]\n        work_pieces = pieces if side == \"bottom\" else [p[::-1, :] for p in pieces]\n\n        widths = sorted({p.shape[1] for p in work_pieces})\n        for w in widths:\n            for c in range(W - w + 1):\n                strip = work_base[:, c : c + w]\n                full_rows = np.where(np.all(strip != 0, axis=1))[0]\n                if len(full_rows) == 0:\n                    continue\n                r_solid = int(full_rows[-1])\n                non_zero_rows = np.where(np.any(strip != 0, axis=1))[0]\n                r_last_wall = int(non_zero_rows[-1])\n\n                # Must be an actual indentation below r_solid\n                if r_last_wall <= r_solid:\n                    continue\n\n                def dfs(cur_strip, r_gap, r_max, used_mask, placements):\n                    for idx, p in enumerate(work_pieces):\n                        if (used_mask >> idx) & 1:\n                            continue\n                        ph, pw = p.shape\n                        if pw != w or r_gap + ph > H:\n                            continue\n                        target = cur_strip[r_gap : r_gap + ph, :]\n                        # No overlap allowed\n                        if np.any((target != 0) & (p != 0)):\n                            continue\n                        # Row r_gap must become completely full\n                        if np.any((target[0, :] == 0) & (p[0, :] == 0)):\n                            continue\n\n                        next_strip = cur_strip.copy()\n                        next_strip[r_gap : r_gap + ph, :] += p\n                        next_max = max(r_max, r_gap + ph - 1)\n                        next_used = used_mask | (1 << idx)\n                        next_placements = placements + [(side, c, r_gap, idx)]\n\n                        # Find next row with a 0 between r_gap and next_max\n                        sub = next_strip[r_gap : next_max + 1, :]\n                        zero_rows = np.where(np.any(sub == 0, axis=1))[0]\n                        if len(zero_rows) == 0:\n                            valid_stacks.append((next_used, next_placements))\n                        else:\n                            dfs(\n                                next_strip,\n                                r_gap + int(zero_rows[0]),\n                                next_max,\n                                next_used,\n                                next_placements,\n                            )\n\n                dfs(strip.copy(), r_solid + 1, r_last_wall, 0, [])\n\n    # Partition all pieces into disjoint valid stacks\n    full_mask = (1 << num_pieces) - 1\n    chosen_placements = []\n\n    def solve_partition(idx, mask, acc):\n        if mask == full_mask:\n            chosen_placements.extend(acc)\n            return True\n        for i in range(idx, len(valid_stacks)):\n            smask, splacements = valid_stacks[i]\n            if (mask & smask) == 0:\n                if solve_partition(i + 1, mask | smask, acc + splacements):\n                    return True\n        return False\n\n    solve_partition(0, 0, [])\n\n    # Render the chosen placements onto the base grid\n    out = base.copy()\n    for side, c, r_gap, idx in chosen_placements:\n        p = pieces[idx]\n        ph, pw = p.shape\n        r_top = r_gap if side == \"bottom\" else H - r_gap - ph\n        mask = p != 0\n        out[r_top : r_top + ph, c : c + pw][mask] = p[mask]\n\n    return out.tolist()\n", "16de56c4.py": "# ARC-AGI-2 task 16de56c4: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n\n    # Rotate the grid so that the starting border is on the left (column 0)\n    rotations = 0\n    for k in range(4):\n        rotated = np.rot90(arr, k)\n        non_zero_rows = [r for r in range(rotated.shape[0]) if np.any(rotated[r] != 0)]\n        if non_zero_rows and all(rotated[r, 0] != 0 for r in non_zero_rows):\n            arr = rotated\n            rotations = k\n            break\n\n    h, w = arr.shape\n    out = arr.copy()\n\n    for r in range(h):\n        start_color = arr[r, 0]\n        if start_color == 0:\n            continue\n\n        # Find step size d from the next pixel matching the start color\n        matches = [c for c in range(1, w) if arr[r, c] == start_color]\n        if not matches:\n            continue\n        d = matches[0]\n\n        # Trace the arithmetic progression 0, d, 2d, ...\n        positions = list(range(0, w, d))\n        fill_color = start_color\n        stop_idx = len(positions)\n\n        for i, c in enumerate(positions):\n            if arr[r, c] != 0 and arr[r, c] != start_color:\n                fill_color = arr[r, c]\n                stop_idx = i + 1\n                break\n\n        for c in positions[:stop_idx]:\n            out[r, c] = fill_color\n\n    # Rotate back to the original orientation\n    out = np.rot90(out, -rotations)\n    return out.tolist()\n", "1818057f.py": "# ARC-AGI-2 task 1818057f: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    out = arr.copy()\n    h, w = arr.shape\n\n    for r in range(1, h - 1):\n        for c in range(1, w - 1):\n            if (\n                arr[r, c] == 4\n                and arr[r - 1, c] == 4\n                and arr[r + 1, c] == 4\n                and arr[r, c - 1] == 4\n                and arr[r, c + 1] == 4\n            ):\n                out[r, c] = 8\n                out[r - 1, c] = 8\n                out[r + 1, c] = 8\n                out[r, c - 1] = 8\n                out[r, c + 1] = 8\n\n    return out.tolist()\n", "195c6913.py": "# ARC-AGI-2 task 195c6913: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n    out = arr.copy()\n\n    # The two most common colours are wall_color and path_color; (0, 0) is wall_color.\n    counts = Counter(arr.flatten())\n    top2 = [c for c, _ in counts.most_common(2)]\n    wall_color = int(arr[0, 0])\n    path_color = top2[1] if top2[0] == wall_color else top2[0]\n\n    # Find 2x2 legend blocks (top-left sequence blocks and bottom-right collision block).\n    top_left_blocks = []\n    hit_color = None\n    visited = np.zeros((H, W), dtype=bool)\n\n    for r in range(H - 1):\n        for c in range(W - 1):\n            val = int(arr[r, c])\n            if val not in (wall_color, path_color) and not visited[r, c]:\n                if (\n                    arr[r, c + 1] == val\n                    and arr[r + 1, c] == val\n                    and arr[r + 1, c + 1] == val\n                ):\n                    visited[r : r + 2, c : c + 2] = True\n                    out[r : r + 2, c : c + 2] = wall_color\n                    if r < H // 2 and c < W // 2:\n                        top_left_blocks.append((c, val))\n                    else:\n                        hit_color = val\n\n    top_left_blocks.sort(key=lambda x: x[0])\n    seq = [val for _, val in top_left_blocks]\n\n    # Find starting pixels in column 0 and trace each path.\n    for r0 in range(H):\n        if arr[r0, 0] not in (wall_color, path_color) and not visited[r0, 0]:\n            r, c = r0, 0\n            dr, dc = 0, 1\n            idx = 0\n            out[r, c] = seq[idx % len(seq)]\n\n            while True:\n                nr, nc = r + dr, c + dc\n                if not (0 <= nr < H and 0 <= nc < W):\n                    break\n                if arr[nr, nc] == path_color:\n                    r, c = nr, nc\n                    idx += 1\n                    out[r, c] = seq[idx % len(seq)]\n                else:\n                    out[nr, nc] = hit_color\n                    dr, dc = (-1, 0) if (dr, dc) == (0, 1) else (0, 1)\n                    if not (\n                        0 <= r + dr < H\n                        and 0 <= c + dc < W\n                        and arr[r + dr, c + dc] == path_color\n                    ):\n                        break\n\n    return out.tolist()\n", "20270e3b.py": "# ARC-AGI-2 task 20270e3b: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import deque\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H, W = len(grid), len(grid[0])\n\n    def get_components(cells, connectivity=4):\n        remaining = set(cells)\n        comps = []\n        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n        if connectivity == 8:\n            dirs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]\n        while remaining:\n            start = next(iter(remaining))\n            remaining.remove(start)\n            comp = {start}\n            q = deque([start])\n            while q:\n                r, c = q.popleft()\n                for dr, dc in dirs:\n                    nr, nc = r + dr, c + dc\n                    if (nr, nc) in remaining:\n                        remaining.remove((nr, nc))\n                        comp.add((nr, nc))\n                        q.append((nr, nc))\n            comps.append(comp)\n        return comps\n\n    sevens = {(r, c) for r in range(H) for c in range(W) if grid[r][c] == 7}\n    non_bg = {(r, c) for r in range(H) for c in range(W) if grid[r][c] != 1}\n\n    s_comps = get_components(sevens, connectivity=4)\n    if len(s_comps) != 2:\n        s_comps = get_components(sevens, connectivity=8)\n    S1, S2 = s_comps[0], s_comps[1]\n\n    pieces = get_components(non_bg, connectivity=4)\n    if len(pieces) != 2:\n        pieces = get_components(non_bg, connectivity=8)\n\n    P1 = set()\n    P2 = set()\n    others = []\n    for comp in pieces:\n        if comp & S1:\n            P1 |= comp\n        elif comp & S2:\n            P2 |= comp\n        else:\n            others.append(comp)\n\n    for comp in others:\n        d1 = min(abs(r1 - r2) + abs(c1 - c2) for r1, c1 in comp for r2, c2 in P1)\n        d2 = min(abs(r1 - r2) + abs(c1 - c2) for r1, c1 in comp for r2, c2 in P2)\n        if d1 <= d2:\n            P1 |= comp\n        else:\n            P2 |= comp\n\n    F1 = P1 - S1\n    F2 = P2 - S2\n\n    r1_min, c1_min = min(S1)\n    r2_min, c2_min = min(S2)\n\n    best_shift = (0, 0)\n    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:\n        if all((r + dr, c + dc) in F1 for r, c in S1) and all(\n            (r - dr, c - dc) in F2 for r, c in S2\n        ):\n            shift = (r1_min + dr - r2_min, c1_min + dc - c2_min)\n            best_shift = shift\n            shifted_F2 = {(r + shift[0], c + shift[1]) for r, c in F2}\n            if not (F1 & shifted_F2):\n                break\n\n    dr_shift, dc_shift = best_shift\n    combined_fours = F1 | {(r + dr_shift, c + dc_shift) for r, c in F2}\n\n    min_r = min(r for r, _ in combined_fours)\n    max_r = max(r for r, _ in combined_fours)\n    min_c = min(c for _, c in combined_fours)\n    max_c = max(c for _, c in combined_fours)\n\n    out_h = max_r - min_r + 1\n    out_w = max_c - min_c + 1\n    out = [[1] * out_w for _ in range(out_h)]\n    for r, c in combined_fours:\n        out[r - min_r][c - min_c] = 4\n\n    return out\n", "21897d95.py": "# ARC-AGI-2 task 21897d95: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import deque\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n    cleaned = arr.copy()\n\n    # 1. Detect arrow markers and restore underlying region colors\n    markers = []\n    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n    for r in range(H):\n        for c in range(W):\n            for dr, dc in dirs:\n                rt, ct = r + dr, c + dc\n                rw1, cw1 = r - dc, c + dr\n                rw2, cw2 = r + dc, c - dr\n                rf1, cf1 = r + dr - dc, c + dc + dr\n                rf2, cf2 = r + dr + dc, c + dc - dr\n\n                coords = [(rt, ct), (rw1, cw1), (rw2, cw2), (rf1, cf1), (rf2, cf2)]\n                if not all(0 <= nr < H and 0 <= nc < W for nr, nc in coords):\n                    continue\n\n                if (\n                    arr[rt, ct] == 1\n                    and arr[rw1, cw1] == 1\n                    and arr[rw2, cw2] == 1\n                    and arr[rf1, cf1] == arr[rf2, cf2]\n                    and arr[rf1, cf1] != 1\n                ):\n                    bg = int(arr[rf1, cf1])\n                    center_val = int(arr[r, c])\n                    payload = bg if center_val == 1 else center_val\n                    markers.append((r, c, dr, dc, bg, payload))\n                    for nr, nc in [(r, c), (rt, ct), (rw1, cw1), (rw2, cw2)]:\n                        cleaned[nr, nc] = bg\n\n    # 2. Detect 3x3 L-shape rotation marker (if present)\n    rot_k = 0  # np.rot90 k: 1 for 90 CCW, -1 for 90 CW\n    visited_l = np.zeros((H, W), dtype=bool)\n    for r in range(H - 2):\n        for c in range(W - 2):\n            sub = cleaned[r : r + 3, c : c + 3]\n            # Top-right L-shape -> 90 deg CCW\n            if (\n                sub[0, 0] == sub[0, 1] == sub[0, 2] == sub[1, 2] == sub[2, 2]\n                and sub[1, 0] == sub[1, 1] == sub[2, 0] == sub[2, 1]\n                and sub[0, 0] != sub[1, 0]\n            ):\n                l_color = sub[0, 0]\n                bg_color = sub[1, 0]\n                # Check that the 5 pixels are an isolated component of l_color\n                q = deque([(r, c)])\n                comp = {(r, c)}\n                while q:\n                    cr, cc = q.popleft()\n                    for dr, dc in dirs:\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and (nr, nc) not in comp and cleaned[nr, nc] == l_color:\n                            comp.add((nr, nc))\n                            q.append((nr, nc))\n                if len(comp) == 5:\n                    rot_k = 1\n                    for nr, nc in comp:\n                        cleaned[nr, nc] = bg_color\n                    break\n\n            # Top-left L-shape -> 90 deg CW\n            if (\n                sub[0, 0] == sub[0, 1] == sub[0, 2] == sub[1, 0] == sub[2, 0]\n                and sub[1, 1] == sub[1, 2] == sub[2, 1] == sub[2, 2]\n                and sub[0, 0] != sub[1, 1]\n            ):\n                l_color = sub[0, 0]\n                bg_color = sub[1, 1]\n                q = deque([(r, c)])\n                comp = {(r, c)}\n                while q:\n                    cr, cc = q.popleft()\n                    for dr, dc in dirs:\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and (nr, nc) not in comp and cleaned[nr, nc] == l_color:\n                            comp.add((nr, nc))\n                            q.append((nr, nc))\n                if len(comp) == 5:\n                    rot_k = -1\n                    for nr, nc in comp:\n                        cleaned[nr, nc] = bg_color\n                    break\n\n    # 3. Label 4-connected regions in the cleaned grid\n    labels = np.full((H, W), -1, dtype=int)\n    region_colors = []\n    num_regions = 0\n    for r in range(H):\n        for c in range(W):\n            if labels[r, c] == -1:\n                color = int(cleaned[r, c])\n                region_colors.append(color)\n                q = deque([(r, c)])\n                labels[r, c] = num_regions\n                while q:\n                    cr, cc = q.popleft()\n                    for dr, dc in dirs:\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and labels[nr, nc] == -1 and cleaned[nr, nc] == color:\n                            labels[nr, nc] = num_regions\n                            q.append((nr, nc))\n                num_regions += 1\n\n    # 4. Apply color transfers from each marker to the target region it points at\n    new_region_colors = list(region_colors)\n    for r, c, dr, dc, bg, payload in markers:\n        src_reg = labels[r, c]\n        nr, nc = r + 2 * dr, c + 2 * dc\n        while 0 <= nr < H and 0 <= nc < W and labels[nr, nc] == src_reg:\n            nr += dr\n            nc += dc\n        if 0 <= nr < H and 0 <= nc < W:\n            target_reg = labels[nr, nc]\n            new_region_colors[target_reg] = payload\n\n    # 5. Build output grid and apply rotation if needed\n    out = np.zeros_like(cleaned)\n    for reg_id, color in enumerate(new_region_colors):\n        out[labels == reg_id] = color\n\n    if rot_k != 0:\n        out = np.rot90(out, rot_k)\n\n    return out.tolist()\n", "221dfab4.py": "# ARC-AGI-2 task 221dfab4: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    vals, counts = np.unique(arr, return_counts=True)\n    bg = int(vals[np.argmax(counts)])\n\n    # Rotate the grid so that the edge containing 4s is at the top (row 0)\n    rot_k = 0\n    for k in range(4):\n        rotated = np.rot90(arr, k)\n        if np.any(rotated[0] == 4):\n            arr = rotated.copy()\n            rot_k = k\n            break\n\n    beam_cols = np.where(arr[0] == 4)[0]\n\n    for r in range(arr.shape[0]):\n        if r % 2 == 1:\n            arr[r, beam_cols] = bg\n        elif r % 6 == 4:\n            arr[r, beam_cols] = 3\n            arr[r, arr[r] != bg] = 3\n        else:\n            arr[r, beam_cols] = 4\n\n    return np.rot90(arr, -rot_k).tolist()\n", "269e22fb.py": "# ARC-AGI-2 task 269e22fb: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\nMASTER = np.array([\n    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],\n    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],\n    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0],\n    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1],\n    [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1],\n    [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],\n    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0],\n    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1],\n    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0],\n    [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],\n    [0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0],\n    [0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],\n    [0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],\n    [0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],\n    [0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0],\n    [0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0],\n    [0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],\n    [1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],\n    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1],\n])\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    inp = np.array(grid)\n    h, w = inp.shape\n    colors = sorted(set(inp.flatten()))\n    if len(colors) != 2:\n        return grid\n\n    color_pairs = [(colors[0], colors[1]), (colors[1], colors[0])]\n\n    for flip in (False, True):\n        base = np.fliplr(MASTER) if flip else MASTER\n        for k in range(4):\n            rotated = np.rot90(base, k)\n            for c0, c1 in color_pairs:\n                candidate = np.where(rotated == 0, c0, c1)\n                for r in range(20 - h + 1):\n                    for c in range(20 - w + 1):\n                        if np.array_equal(candidate[r : r + h, c : c + w], inp):\n                            return candidate.tolist()\n\n    return grid\n", "271d71e2.py": "# ARC-AGI-2 task 271d71e2: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    out = arr.copy()\n    H, W = arr.shape\n\n    visited = np.zeros((H, W), dtype=bool)\n    boxes = []\n\n    for r in range(H):\n        for c in range(W):\n            if arr[r, c] == 0 and not visited[r, c]:\n                # Find bounding box of this 0-bordered rectangle\n                r2 = r\n                while r2 + 1 < H and arr[r2 + 1, c] == 0:\n                    r2 += 1\n                c2 = c\n                while c2 + 1 < W and arr[r, c2 + 1] == 0:\n                    c2 += 1\n                visited[r : r2 + 1, c : c2 + 1] = True\n                boxes.append((r, r2, c, c2))\n\n    for r1, r2, c1, c2 in boxes:\n        interior = arr[r1 + 1 : r2, c1 + 1 : c2]\n        num_5 = int(np.sum(interior == 5))\n        num_7 = int(np.sum(interior == 7))\n\n        # Determine movement direction and distance between the two 9-bars\n        if r1 > 0 and np.all(arr[r1 - 1, c1 : c2 + 1] == 9):\n            direction = \"UP\"\n            r_adj = r1 - 1\n            r_targ = r_adj - 1\n            while not np.all(arr[r_targ, c1 : c2 + 1] == 9):\n                r_targ -= 1\n            dist = r_adj - r_targ\n            dr, dc = -1, 0\n            rot_k = 0\n        elif r2 + 1 < H and np.all(arr[r2 + 1, c1 : c2 + 1] == 9):\n            direction = \"DOWN\"\n            r_adj = r2 + 1\n            r_targ = r_adj + 1\n            while not np.all(arr[r_targ, c1 : c2 + 1] == 9):\n                r_targ += 1\n            dist = r_targ - r_adj\n            dr, dc = 1, 0\n            rot_k = 2\n        elif c1 > 0 and np.all(arr[r1 : r2 + 1, c1 - 1] == 9):\n            direction = \"LEFT\"\n            c_adj = c1 - 1\n            c_targ = c_adj - 1\n            while not np.all(arr[r1 : r2 + 1, c_targ] == 9):\n                c_targ -= 1\n            dist = c_adj - c_targ\n            dr, dc = 0, -1\n            rot_k = 3\n        elif c2 + 1 < W and np.all(arr[r1 : r2 + 1, c2 + 1] == 9):\n            direction = \"RIGHT\"\n            c_adj = c2 + 1\n            c_targ = c_adj + 1\n            while not np.all(arr[r1 : r2 + 1, c_targ] == 9):\n                c_targ += 1\n            dist = c_targ - c_adj\n            dr, dc = 0, 1\n            rot_k = 1\n        else:\n            continue\n\n        steps = min(dist, num_5)\n        total_7 = num_7 + steps\n\n        # Construct the updated interior in canonical orientation (facing UP)\n        rot_int = np.rot90(interior, k=rot_k)\n        flat = np.full(rot_int.size, 5, dtype=int)\n        flat[:total_7] = 7\n        new_rot_int = flat.reshape(rot_int.shape)\n        new_interior = np.rot90(new_rot_int, k=-rot_k)\n\n        # Clear original box and its adjacent 9-bar\n        out[r1 : r2 + 1, c1 : c2 + 1] = 6\n        if direction == \"UP\":\n            out[r1 - 1, c1 : c2 + 1] = 6\n        elif direction == \"DOWN\":\n            out[r2 + 1, c1 : c2 + 1] = 6\n        elif direction == \"LEFT\":\n            out[r1 : r2 + 1, c1 - 1] = 6\n        elif direction == \"RIGHT\":\n            out[r1 : r2 + 1, c2 + 1] = 6\n\n        # Draw moved box and moved 9-bar\n        nr1, nr2 = r1 + steps * dr, r2 + steps * dr\n        nc1, nc2 = c1 + steps * dc, c2 + steps * dc\n        out[nr1 : nr2 + 1, nc1 : nc2 + 1] = 0\n        out[nr1 + 1 : nr2, nc1 + 1 : nc2] = new_interior\n\n        if direction == \"UP\":\n            out[nr1 - 1, nc1 : nc2 + 1] = 9\n        elif direction == \"DOWN\":\n            out[nr2 + 1, nc1 : nc2 + 1] = 9\n        elif direction == \"LEFT\":\n            out[nr1 : nr2 + 1, nc1 - 1] = 9\n        elif direction == \"RIGHT\":\n            out[nr1 : nr2 + 1, nc2 + 1] = 9\n\n    return out.tolist()\n", "28a6681f.py": "# ARC-AGI-2 task 28a6681f: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    W = len(grid[0])\n\n    # Keep all static obstacles and remove water (1s) to simulate settling\n    out = [[0 if cell == 1 else cell for cell in row] for row in grid]\n\n    # Process water units from bottom to top\n    water_positions = [\n        (r, c)\n        for r in range(H - 1, -1, -1)\n        for c in range(W)\n        if grid[r][c] == 1\n    ]\n\n    for start_r, start_c in water_positions:\n        r, c = start_r, start_c\n        while True:\n            # Fall straight down while possible\n            while r + 1 < H and out[r + 1][c] == 0:\n                r += 1\n\n            # Scan left and right for a drop-off\n            left_drop = None\n            left_bound = c\n            for cl in range(c - 1, -1, -1):\n                if out[r][cl] != 0:\n                    break\n                left_bound = cl\n                if r + 1 < H and out[r + 1][cl] == 0:\n                    left_drop = cl\n                    break\n\n            right_drop = None\n            right_bound = c\n            for cr in range(c + 1, W):\n                if out[r][cr] != 0:\n                    break\n                right_bound = cr\n                if r + 1 < H and out[r + 1][cr] == 0:\n                    right_drop = cr\n                    break\n\n            if left_drop is not None and right_drop is not None:\n                c = left_drop if (c - left_drop <= right_drop - c) else right_drop\n            elif left_drop is not None:\n                c = left_drop\n            elif right_drop is not None:\n                c = right_drop\n            else:\n                # Settle inside the enclosed horizontal basin at row r\n                candidates = [\n                    col for col in range(left_bound, right_bound + 1)\n                    if out[r][col] == 0\n                ]\n                target_c = max(candidates, key=lambda col: abs(col - c))\n                out[r][target_c] = 1\n                break\n\n    return out\n", "291dc1e1.py": "# ARC-AGI-2 task 291dc1e1: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    H, W = arr.shape\n\n    # Locate the '0' corner on the border\n    corners = [(0, 0), (0, W - 1), (H - 1, 0), (H - 1, W - 1)]\n    r0, c0 = next((r, c) for r, c in corners if arr[r, c] == 0)\n\n    # Determine whether the '1'-axis is horizontal (along row r0) or vertical (along col c0)\n    adj_r = 1 if r0 == 0 else H - 2\n    adj_c = 1 if c0 == 0 else W - 2\n    one_is_horizontal = (arr[r0, adj_c] == 1)\n\n    # Extract interior slices (excluding the border row r0 and border col c0)\n    rows = [r for r in range(H) if r != r0]\n    cols = [c for c in range(W) if c != c0]\n\n    objects = []\n\n    if one_is_horizontal:\n        # Tracks are horizontal bands of rows (indexed along the vertical '2'-axis)\n        r_idx = 0\n        tracks = []\n        while r_idx < len(rows):\n            r = rows[r_idx]\n            if np.any(arr[r, cols] != 8):\n                track_rows = [r]\n                r_idx += 1\n                while r_idx < len(rows) and np.any(arr[rows[r_idx], cols] != 8):\n                    track_rows.append(rows[r_idx])\n                    r_idx += 1\n                tracks.append(track_rows)\n            else:\n                r_idx += 1\n\n        if r0 == H - 1:\n            tracks.reverse()\n\n        for track_rows in tracks:\n            c_idx = 0\n            track_objs = []\n            while c_idx < len(cols):\n                c = cols[c_idx]\n                if np.any(arr[track_rows, c] != 8):\n                    obj_cols = [c]\n                    c_idx += 1\n                    while c_idx < len(cols) and np.any(arr[track_rows, cols[c_idx]] != 8):\n                        obj_cols.append(cols[c_idx])\n                        c_idx += 1\n                    obj = arr[np.ix_(track_rows, obj_cols)]\n                    track_objs.append(obj)\n                else:\n                    c_idx += 1\n            if c0 == W - 1:\n                track_objs.reverse()\n            objects.extend(track_objs)\n    else:\n        # Tracks are vertical bands of columns (indexed along the horizontal '2'-axis)\n        c_idx = 0\n        tracks = []\n        while c_idx < len(cols):\n            c = cols[c_idx]\n            if np.any(arr[rows, c] != 8):\n                track_cols = [c]\n                c_idx += 1\n                while c_idx < len(cols) and np.any(arr[rows, cols[c_idx]] != 8):\n                    track_cols.append(cols[c_idx])\n                    c_idx += 1\n                tracks.append(track_cols)\n            else:\n                c_idx += 1\n\n        if c0 == W - 1:\n            tracks.reverse()\n\n        for track_cols in tracks:\n            r_idx = 0\n            track_objs = []\n            while r_idx < len(rows):\n                r = rows[r_idx]\n                if np.any(arr[r, track_cols] != 8):\n                    obj_rows = [r]\n                    r_idx += 1\n                    while r_idx < len(rows) and np.any(arr[rows[r_idx], track_cols] != 8):\n                        obj_rows.append(rows[r_idx])\n                        r_idx += 1\n                    obj = arr[np.ix_(obj_rows, track_cols)]\n                    # Rotate 90 degrees counter-clockwise to make the object horizontal\n                    track_objs.append(np.rot90(obj, 1))\n                else:\n                    r_idx += 1\n            if r0 == H - 1:\n                track_objs.reverse()\n            objects.extend(track_objs)\n\n    max_w = max(obj.shape[1] for obj in objects)\n    out_strips = []\n    for obj in objects:\n        h, w = obj.shape\n        strip = np.full((h, max_w), 8, dtype=int)\n        pad = (max_w - w) // 2\n        strip[:, pad:pad + w] = obj\n        out_strips.append(strip)\n\n    return np.vstack(out_strips).tolist()\n", "2b83f449.py": "# ARC-AGI-2 task 2b83f449: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    W = len(grid[0])\n    out = [row[:] for row in grid]\n\n    starts = []\n    for r in range(0, H, 2):\n        for c in range(W):\n            if out[r][c] == 3:\n                starts.append((r, c))\n                out[r][c] = 8\n\n    down_bridges = set()\n    for r in range(1, H, 2):\n        for c in range(1, W - 1):\n            if grid[r][c - 1] == 7 and grid[r][c] == 7 and grid[r][c + 1] == 7:\n                out[r][c - 1] = 8\n                out[r][c] = 6\n                out[r][c + 1] = 8\n                out[r - 1][c] = 6\n                out[r + 1][c] = 6\n                down_bridges.add((r - 1, c))\n\n    for r_start, c_start in starts:\n        direction = 1 if c_start == 0 else -1\n        curr_r, curr_c, curr_dir = r_start, c_start, direction\n        moved_down = False\n\n        while True:\n            next_c = curr_c + curr_dir\n            while 0 <= next_c < W and out[curr_r][next_c] not in (0, 6):\n                next_c += curr_dir\n\n            if 0 <= next_c < W and (curr_r, next_c) in down_bridges:\n                curr_r += 2\n                curr_c = next_c\n                curr_dir = -curr_dir\n                moved_down = True\n            else:\n                if not moved_down:\n                     place_c = c_start\n                     step = direction\n                else:\n                    place_c = next_c - curr_dir\n                    step = -curr_dir\n\n                while out[curr_r][place_c] == 3:\n                    place_c += step\n                out[curr_r][place_c] = 3\n                break\n\n    return out\n", "2c181942.py": "# ARC-AGI-2 task 2c181942: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    rows, cols = arr.shape\n    vals, counts = np.unique(arr, return_counts=True)\n    bg = int(vals[np.argmax(counts)])\n\n    # 1. Locate the 4x4 multi-colored central ring\n    r0 = c0 = -1\n    for r in range(rows - 3):\n        for c in range(cols - 3):\n            sub = arr[r:r+4, c:c+4]\n            top = sub[0, 1]\n            bot = sub[3, 1]\n            left = sub[1, 0]\n            right = sub[1, 3]\n            if (top != bg and sub[0, 2] == top and\n                bot != bg and sub[3, 2] == bot and\n                left != bg and sub[2, 0] == left and\n                right != bg and sub[2, 3] == right):\n                empty_spots = [\n                    sub[0, 0], sub[0, 3], sub[3, 0], sub[3, 3],\n                    sub[1, 1], sub[1, 2], sub[2, 1], sub[2, 2]\n                ]\n                if all(x == bg for x in empty_spots) and len({top, bot, left, right}) > 1:\n                    r0, c0 = r, c\n                    break\n        if r0 != -1:\n            break\n\n    out = np.full_like(arr, bg)\n    out[r0:r0+4, c0:c0+4] = arr[r0:r0+4, c0:c0+4]\n\n    # Mask out the central ring to find the scattered shapes\n    work = arr.copy()\n    work[r0:r0+4, c0:c0+4] = bg\n\n    def is_connector(edge: np.ndarray, color: int) -> bool:\n        n = len(edge)\n        if n % 2 != 0:\n            return False\n        expected = np.full(n, bg)\n        expected[n // 2 - 1 : n // 2 + 1] = color\n        return np.array_equal(edge, expected)\n\n    sides = [\n        (\"top\", int(arr[r0, c0 + 1])),\n        (\"bot\", int(arr[r0 + 3, c0 + 1])),\n        (\"left\", int(arr[r0 + 1, c0])),\n        (\"right\", int(arr[r0 + 1, c0 + 3])),\n    ]\n\n    # 2. Rotate and attach each matching shape to its corresponding ring side\n    for direction, color in sides:\n        pts = np.argwhere(work == color)\n        if len(pts) == 0:\n            continue\n        rmin, cmin = pts.min(axis=0)\n        rmax, cmax = pts.max(axis=0)\n        shape = np.where(work[rmin:rmax+1, cmin:cmax+1] == color, color, bg)\n\n        for k in range(4):\n            rot = np.rot90(shape, k)\n            h, w = rot.shape\n            if direction == \"top\" and is_connector(rot[-1, :], color):\n                sr, sc = r0 - h, c0 + 1 - (w // 2 - 1)\n                break\n            if direction == \"bot\" and is_connector(rot[0, :], color):\n                sr, sc = r0 + 4, c0 + 1 - (w // 2 - 1)\n                break\n            if direction == \"left\" and is_connector(rot[:, -1], color):\n                sr, sc = r0 + 1 - (h // 2 - 1), c0 - w\n                break\n            if direction == \"right\" and is_connector(rot[:, 0], color):\n                sr, sc = r0 + 1 - (h // 2 - 1), c0 + 4\n                break\n\n        mask = rot == color\n        out[sr:sr+h, sc:sc+w][mask] = color\n\n    return out.tolist()\n", "2d0172a1.py": "# ARC-AGI-2 task 2d0172a1: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter, deque\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n\n    counts = Counter(arr.flatten())\n    bg = counts.most_common(1)[0][0]\n    fg = [c for c in counts if c != bg][0]\n\n    # Find 4-connected components of foreground pixels\n    visited = np.zeros((H, W), dtype=bool)\n    components = []\n    for r in range(H):\n        for c in range(W):\n            if arr[r, c] == fg and not visited[r, c]:\n                comp = []\n                q = deque([(r, c)])\n                visited[r, c] = True\n                while q:\n                    cr, cc = q.popleft()\n                    comp.append((cr, cc))\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and arr[nr, nc] == fg and not visited[nr, nc]:\n                            visited[nr, nc] = True\n                            q.append((nr, nc))\n                components.append(comp)\n\n    # Determine the filled interior of each component to separate loops from blobs\n    loops = []\n    blobs = []\n    for comp in components:\n        padded = np.zeros((H + 2, W + 2), dtype=bool)\n        for r, c in comp:\n            padded[r + 1, c + 1] = True\n\n        ext = np.zeros((H + 2, W + 2), dtype=bool)\n        q = deque([(0, 0)])\n        ext[0, 0] = True\n        while q:\n            cr, cc = q.popleft()\n            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                nr, nc = cr + dr, cc + dc\n                if 0 <= nr < H + 2 and 0 <= nc < W + 2 and not padded[nr, nc] and not ext[nr, nc]:\n                    ext[nr, nc] = True\n                    q.append((nr, nc))\n\n        filled = set()\n        for r in range(H):\n            for c in range(W):\n                if not ext[r + 1, c + 1]:\n                    filled.add((r, c))\n\n        if len(filled) > len(comp):\n            loops.append({\"comp\": set(comp), \"filled\": filled})\n        else:\n            rs = [r for r, _ in comp]\n            cs = [c for _, c in comp]\n            blobs.append({\n                \"comp\": set(comp),\n                \"r_span\": (min(rs), max(rs)),\n                \"c_span\": (min(cs), max(cs)),\n                \"rep\": comp[0],\n            })\n\n    # Cluster intervals by overlap to assign logical rows and columns to blobs\n    def cluster_intervals(intervals):\n        sorted_ivs = sorted(intervals, key=lambda x: (x[0], x[1]))\n        merged = []\n        for iv in sorted_ivs:\n            if not merged or iv[0] > merged[-1][1]:\n                merged.append([iv[0], iv[1]])\n            else:\n                merged[-1][1] = max(merged[-1][1], iv[1])\n        return merged\n\n    row_clusters = cluster_intervals([b[\"r_span\"] for b in blobs])\n    col_clusters = cluster_intervals([b[\"c_span\"] for b in blobs])\n\n    for b in blobs:\n        r0, r1 = b[\"r_span\"]\n        c0, c1 = b[\"c_span\"]\n        b[\"l_row\"] = next(i for i, ( mr0, mr1) in enumerate(row_clusters) if max(r0, mr0) <= min(r1, mr1))\n        b[\"l_col\"] = next(i for i, (mc0, mc1) in enumerate(col_clusters) if max(c0, mc0) <= min(c1, mc1))\n\n    # Determine logical bounding boxes for each loop\n    for idx, loop in enumerate(loops):\n        loop[\"id\"] = idx\n        loop[\"area\"] = len(loop[\"filled\"])\n        enclosed = [b for b in blobs if b[\"rep\"] in loop[\"filled\"]]\n        loop[\"r_min\"] = min(b[\"l_row\"] for b in enclosed)\n        loop[\"r_max\"] = max(b[\"l_row\"] for b in enclosed)\n        loop[\"c_min\"] = min(b[\"l_col\"] for b in enclosed)\n        loop[\"c_max\"] = max(b[\"l_col\"] for b in enclosed)\n\n    # Build 1D coordinate mappings for rows and columns\n    def build_axis(num_logical, min_key, max_key):\n        events = []\n        for i in range(num_logical):\n            starts = [L for L in loops if L[min_key] == i]\n            starts.sort(key=lambda L: -L[\"area\"])\n            for L in starts:\n                events.append((\"start\", L[\"id\"]))\n\n            events.append((\"point\", i))\n\n            ends = [L for L in loops if L[max_key] == i]\n            ends.sort(key=lambda L: L[\"area\"])\n            for L in ends:\n                events.append((\"end\", L[\"id\"]))\n\n        offset = 1 if events[0][0] == \"point\" else 0\n        start_pos = {}\n        end_pos = {}\n        point_pos = {}\n        for idx, (etype, eid) in enumerate(events):\n            pos = offset + 2 * idx\n            if etype == \"start\":\n                start_pos[eid] = pos\n            elif etype == \"end\":\n                end_pos[eid] = pos\n            else:\n                point_pos[eid] = pos\n\n        total_len = offset + 2 * len(events) - 1 + (1 if events[-1][0] == \"point\" else 0)\n        return total_len, start_pos, end_pos, point_pos\n\n    out_H, r_start, r_end, r_point = build_axis(len(row_clusters), \"r_min\", \"r_max\")\n    out_W, c_start, c_end, c_point = build_axis(len(col_clusters), \"c_min\", \"c_max\")\n\n    # Render the canonical output grid\n    out = np.full((out_H, out_W), bg, dtype=int)\n    for L in loops:\n        lid = L[\"id\"]\n        r0, r1 = r_start[lid], r_end[lid]\n        c0, c1 = c_start[lid], c_end[lid]\n        out[r0, c0 : c1 + 1] = fg\n        out[r1, c0 : c1 + 1] = fg\n        out[r0 : r1 + 1, c0] = fg\n        out[r0 : r1 + 1, c1] = fg\n\n    for b in blobs:\n        out[r_point[b[\"l_row\"]], c_point[b[\"l_col\"]]] = fg\n\n    return out.tolist()\n", "31f7f899.py": "# ARC-AGI-2 task 31f7f899: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n    bg_color = 8\n    spine_color = 6\n\n    # Find the central horizontal spine row (the row containing spine_color)\n    mid_r = H // 2\n    for r in range(H):\n        if spine_color in arr[r]:\n            mid_r = r\n            break\n\n    # Group contiguous columns of the same non-spine color into bars\n    bars = []\n    c = 0\n    while c < W:\n        color = int(arr[mid_r, c])\n        if color != spine_color and color != bg_color:\n            c_start = c\n            height = int(np.sum(arr[:, c] != bg_color))\n            while (\n                c + 1 < W\n                and arr[mid_r, c + 1] == color\n                and int(np.sum(arr[:, c + 1] != bg_color)) == height\n            ):\n                c += 1\n            bars.append((c_start, c, color, height))\n        c += 1\n\n    # Sort the heights of the bars in ascending order from left to right\n    sorted_heights = sorted(b[3] for b in bars)\n\n    # Build the output grid\n    out = np.full((H, W), bg_color, dtype=int)\n    out[mid_r, :] = arr[mid_r, :]\n\n    for (c_start, c_end, color, _), new_h in zip(bars, sorted_heights):\n        half = new_h // 2\n        out[mid_r - half : mid_r + half + 1, c_start : c_end + 1] = color\n\n    return out.tolist()\n", "31f7f899_2.py": "# ARC-AGI-2 task 31f7f899: verified on all train pairs\n# model=gemini-4-argon sample=2 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    bg = 8\n    axis_color = 6\n\n    # Find the horizontal axis row (the row containing the axis color 6)\n    axis_rows = np.where((arr == axis_color).any(axis=1))[0]\n    r_axis = int(axis_rows[0])\n\n    # Identify each vertical bar as a contiguous run of columns on the axis row\n    # with the same non-axis color and the same vertical span.\n    bars = []\n    cols = arr.shape[1]\n    c = 0\n    while c < cols:\n        color = int(arr[r_axis, c])\n        if color == axis_color or color == bg:\n            c += 1\n            continue\n\n        non_bg_rows = np.where(arr[:, c] != bg)[0]\n        span = (int(non_bg_rows[0]), int(non_bg_rows[-1]))\n\n        c_end = c\n        while c_end + 1 < cols and int(arr[r_axis, c_end + 1]) == color:\n            next_rows = np.where(arr[:, c_end + 1] != bg)[0]\n            next_span = (int(next_rows[0]), int(next_rows[-1]))\n            if next_span == span:\n                c_end += 1\n            else:\n                break\n\n        bars.append((c, c_end, color, span))\n        c = c_end + 1\n\n    # Sort the vertical spans by total height (ascending)\n    sorted_spans = sorted([b[3] for b in bars], key=lambda s: s[1] - s[0])\n\n    # Construct the output grid\n    out = np.full_like(arr, bg)\n    out[r_axis, :] = arr[r_axis, :]\n\n    for (c_start, c_end, color, _), (r_start, r_end) in zip(bars, sorted_spans):\n        out[r_start : r_end + 1, c_start : c_end + 1] = color\n\n    return out.tolist()\n", "332f06d7.py": "# ARC-AGI-2 task 332f06d7: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import deque\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n\n    zeros = np.argwhere(arr == 0)\n    r0, c0 = zeros.min(axis=0)\n    r1, c1 = zeros.max(axis=0)\n    k = r1 - r0 + 1\n\n    # Free cells are anything other than wall (3)\n    free = arr != 3\n\n    # Compute shortest-path distance from every free cell to the target (2)\n    dist_to_2 = np.full((H, W), np.inf)\n    q = deque()\n    for r, c in np.argwhere(arr == 2):\n        dist_to_2[r, c] = 0\n        q.append((r, c))\n\n    while q:\n        r, c = q.popleft()\n        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < H and 0 <= nc < W and free[nr, nc]:\n                if dist_to_2[nr, nc] == np.inf:\n                    dist_to_2[nr, nc] = dist_to_2[r, c] + 1\n                    q.append((nr, nc))\n\n    # Precompute valid top-left positions for a k x k block\n    valid = np.zeros((H - k + 1, W - k + 1), dtype=bool)\n    for r in range(H - k + 1):\n        for c in range(W - k + 1):\n            valid[r, c] = free[r : r + k, c : c + k].all()\n\n    # BFS to find all k x k positions reachable from (r0, c0)\n    visited = np.zeros_like(valid, dtype=bool)\n    visited[r0, c0] = True\n    q = deque([(r0, c0)])\n    reachable = []\n\n    while q:\n        r, c = q.popleft()\n        reachable.append((r, c))\n        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < H - k + 1 and 0 <= nc < W - k + 1:\n                if valid[nr, nc] and not visited[nr, nc]:\n                    visited[nr, nc] = True\n                    q.append((nr, nc))\n\n    # Pick the reachable position that minimizes the distance to the 2s\n    best_r, best_c = min(\n        reachable,\n        key=lambda pos: dist_to_2[\n            pos[0] : pos[0] + k, pos[1] : pos[1] + k\n        ].sum(),\n    )\n\n    out = arr.copy()\n    out[r0 : r0 + k, c0 : c0 + k] = 1\n    out[best_r : best_r + k, best_c : best_c + k] = 0\n    return out.tolist()\n", "35ab12c3.py": "# ARC-AGI-2 task 35ab12c3: verified on all train pairs\n# model=gemini-4-argon sample=1 round=1 test_correct=1.0\n\nfrom itertools import permutations\nimport numpy as np\n\n\ndef get_line_pixels(\n    p1: tuple[int, int], p2: tuple[int, int]\n) -> list[tuple[int, int]]:\n    r1, c1 = p1\n    r2, c2 = p2\n    steps = max(abs(r2 - r1), abs(c2 - c1))\n    if steps == 0:\n        return [(r1, c1)]\n    dr = (r2 - r1) // steps\n    dc = (c2 - c1) // steps\n    return [(r1 + i * dr, c1 + i * dc) for i in range(steps + 1)]\n\n\ndef build_shape_pixels(points: list[tuple[int, int]]) -> set[tuple[int, int]]:\n    n = len(points)\n    if n == 1:\n        return set(points)\n\n    # Group points along horizontal, vertical, and both 45-degree diagonal lines\n    lines = {}\n    for idx, (r, c) in enumerate(points):\n        for key in [(\"h\", r), (\"v\", c), (\"d1\", r + c), (\"d2\", r - c)]:\n            lines.setdefault(key, []).append(idx)\n\n    # Build adjacency graph of collinear consecutive points\n    adj = {i: {} for i in range(n)}\n    for key, idxs in lines.items():\n        if len(idxs) < 2:\n            continue\n        if key[0] == \"h\":\n            idxs.sort(key=lambda i: points[i][1])\n        else:\n            idxs.sort(key=lambda i: points[i][0])\n        for a, b in zip(idxs, idxs[1:]):\n            r1, c1 = points[a]\n            r2, c2 = points[b]\n            dist = max(abs(r1 - r2), abs(c1 - c2))\n            adj[a][b] = dist\n            adj[b][a] = dist\n\n    # Find connected components and select valid edges\n    visited = set()\n    selected_edges = []\n    for start in range(n):\n        if start in visited:\n            continue\n        comp = []\n        queue = [start]\n        visited.add(start)\n        while queue:\n            curr = queue.pop(0)\n            comp.append(curr)\n            for nxt in adj[curr]:\n                if nxt not in visited:\n                    visited.add(nxt)\n                    queue.append(nxt)\n\n        max_deg = max(len(adj[u]) for u in comp)\n        if max_deg <= 2:\n            for u in comp:\n                for v in adj[u]:\n                    if u < v:\n                        selected_edges.append((u, v))\n        else:\n            # Find the shortest Hamiltonian cycle on the component to eliminate internal chords\n            first = comp[0]\n            rest = comp[1:]\n            best_cost = float(\"inf\")\n            best_cycle = None\n            for perm in permutations(rest):\n                cycle = (first,) + perm\n                cost = 0\n                valid = True\n                for i in range(len(cycle)):\n                    u = cycle[i]\n                    v = cycle[(i + 1) % len(cycle)]\n                    if v not in adj[u]:\n                        valid = False\n                        break\n                    cost += adj[u][v]\n                if valid and cost < best_cost:\n                    best_cost = cost\n                    best_cycle = cycle\n\n            if best_cycle is not None:\n                for i in range(len(best_cycle)):\n                    u = best_cycle[i]\n                    v = best_cycle[(i + 1) % len(best_cycle)]\n                    selected_edges.append((u, v))\n\n    shape_pixels = set(points)\n    for u, v in selected_edges:\n        shape_pixels.update(get_line_pixels(points[u], points[v]))\n    return shape_pixels\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    rows, cols = arr.shape\n    output = np.zeros_like(arr)\n\n    colors = [int(c) for c in np.unique(arr) if c != 0]\n    color_points = {\n        c: [tuple(map(int, pt)) for pt in np.argwhere(arr == c)] for c in colors\n    }\n\n    multi_colors = [c for c in colors if len(color_points[c]) > 1]\n    single_colors = [c for c in colors if len(color_points[c]) == 1]\n\n    # Build shapes for multi-pixel colors\n    shapes = {}\n    for c in multi_colors:\n        shapes[c] = build_shape_pixels(color_points[c])\n\n    # Draw shifted shapes for single-pixel colors first (underneath)\n    for c in single_colors:\n        sr, sc = color_points[c][0]\n        matched = False\n        for mc in multi_colors:\n            for mr, mc_col in color_points[mc]:\n                if max(abs(sr - mr), abs(sc - mc_col)) == 1:\n                    dr, dc = sr - mr, sc - mc_col\n                    for r, col in shapes[mc]:\n                        nr, nc = r + dr, col + dc\n                        if 0 <= nr < rows and 0 <= nc < cols:\n                            output[nr, nc] = c\n                    matched = True\n                    break\n            if matched:\n                break\n        if not matched:\n            output[sr, sc] = c\n\n    # Draw main multi-pixel shapes on top\n    for c in multi_colors:\n        for r, col in shapes[c]:\n            output[r, col] = c\n\n    return output.tolist()\n", "36a08778.py": "# ARC-AGI-2 task 36a08778: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    res = [row[:] for row in grid]\n    h = len(res)\n    w = len(res[0])\n\n    for r in range(h - 1):\n        # Find current stream positions in row r that are blocked from below\n        sources = [c for c in range(w) if res[r][c] == 6 and res[r + 1][c] == 2]\n        for c in sources:\n            # Spread left\n            for c_left in range(c - 1, -1, -1):\n                if res[r][c_left] == 2:\n                    break\n                res[r][c_left] = 6\n                if res[r + 1][c_left] != 2:\n                    break\n            # Spread right\n            for c_right in range(c + 1, w):\n                if res[r][c_right] == 2:\n                    break\n                res[r][c_right] = 6\n                if res[r + 1][c_right] != 2:\n                    break\n\n        # Flow downward into row r + 1 where unblocked\n        for c in range(w):\n            if res[r][c] == 6 and res[r + 1][c] == 7:\n                res[r + 1][c] = 6\n\n    return res\n", "38007db0.py": "# ARC-AGI-2 task 38007db0: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    W = len(grid[0])\n    sep_color = grid[0][0]\n\n    sep_rows = [r for r in range(H) if all(grid[r][c] == sep_color for c in range(W))]\n    sep_cols = [c for c in range(W) if all(grid[r][c] == sep_color for r in range(H))]\n\n    sub_w = sep_cols[1] - sep_cols[0] - 1\n    out_w = sub_w + 2\n    out = [[sep_color] * out_w]\n\n    for i in range(len(sep_rows) - 1):\n        r1, r2 = sep_rows[i] + 1, sep_rows[i + 1]\n        subgrids = []\n        for j in range(len(sep_cols) - 1):\n            c1, c2 = sep_cols[j] + 1, sep_cols[j + 1]\n            sub = tuple(tuple(grid[r][c1:c2]) for r in range(r1, r2))\n            subgrids.append(sub)\n\n        counts = Counter(subgrids)\n        unique_sub = next(sub for sub, cnt in counts.items() if cnt == 1)\n\n        for row in unique_sub:\n            out.append([sep_color] + list(row) + [sep_color])\n        out.append([sep_color] * out_w)\n\n    return out\n", "3a25b0d8.py": "# ARC-AGI-2 task 3a25b0d8: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter, deque\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    H, W = arr.shape\n\n    # 1. Identify background color (most frequent)\n    counts = Counter(arr.flatten().tolist())\n    bg = counts.most_common(1)[0][0]\n\n    # 2. Find 8-connected components of non-background cells\n    visited = np.zeros((H, W), dtype=bool)\n    comps = []\n    for r in range(H):\n        for c in range(W):\n            if arr[r, c] != bg and not visited[r, c]:\n                q = deque([(r, c)])\n                visited[r, c] = True\n                cells = []\n                while q:\n                    cr, cc = q.popleft()\n                    cells.append((cr, cc))\n                    for dr in (-1, 0, 1):\n                        for dc in (-1, 0, 1):\n                            if dr == 0 and dc == 0:\n                                continue\n                            nr, nc = cr + dr, cc + dc\n                            if 0 <= nr < H and 0 <= nc < W and arr[nr, nc] != bg and not visited[nr, nc]:\n                                visited[nr, nc] = True\n                                q.append((nr, nc))\n                comps.append(cells)\n\n    # Separate into wireframe component (single color) and colored component (multiple colors)\n    mono_comps = [comp for comp in comps if len({arr[r, c] for r, c in comp}) == 1]\n    poly_comps = [comp for comp in comps if len({arr[r, c] for r, c in comp}) > 1]\n\n    wireframe_cells = max(mono_comps, key=len)\n    border_color = int(arr[wireframe_cells[0][0], wireframe_cells[0][1]])\n    colored_cells = [cell for comp in poly_comps for cell in comp]\n\n    # 3. Extract bounding boxes for the wireframe and colored figures\n    w_rs = [r for r, _ in wireframe_cells]\n    w_cs = [c for _, c in wireframe_cells]\n    w_rmin, w_rmax, w_cmin, w_cmax = min(w_rs), max(w_rs), min(w_cs), max(w_cs)\n    w_h, w_w = w_rmax - w_rmin + 1, w_cmax - w_cmin + 1\n\n    out = np.full((w_h, w_w), bg, dtype=int)\n    for r, c in wireframe_cells:\n        out[r - w_rmin, c - w_cmin] = border_color\n\n    c_rs = [r for r, _ in colored_cells]\n    c_cs = [c for _, c in colored_cells]\n    c_rmin, c_rmax, c_cmin, c_cmax = min(c_rs), max(c_rs), min(c_cs), max(c_cs)\n    c_h, c_w = c_rmax - c_rmin + 1, c_cmax - c_cmin + 1\n    c_sub = arr[c_rmin:c_rmax + 1, c_cmin:c_cmax + 1]\n\n    # 4. Find enclosed 4-connected background holes in the wireframe\n    padded = np.full((w_h + 2, w_w + 2), bg, dtype=int)\n    padded[1:-1, 1:-1] = out\n    ext_vis = np.zeros((w_h + 2, w_w + 2), dtype=bool)\n    q = deque([(0, 0)])\n    ext_vis[0, 0] = True\n    while q:\n        cr, cc = q.popleft()\n        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n            nr, nc = cr + dr, cc + dc\n            if 0 <= nr < w_h + 2 and 0 <= nc < w_w + 2 and not ext_vis[nr, nc] and padded[nr, nc] == bg:\n                ext_vis[nr, nc] = True\n                q.append((nr, nc))\n\n    hole_vis = ext_vis[1:-1, 1:-1].copy()\n    holes = []\n    for r in range(w_h):\n        for c in range(w_w):\n            if out[r, c] == bg and not hole_vis[r, c]:\n                q = deque([(r, c)])\n                hole_vis[r, c] = True\n                cells = []\n                while q:\n                    cr, cc = q.popleft()\n                    cells.append((cr, cc))\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < w_h and 0 <= nc < w_w and out[nr, nc] == bg and not hole_vis[nr, nc]:\n                            hole_vis[nr, nc] = True\n                            q.append((nr, nc))\n                holes.append(cells)\n\n    # 5. Find 4-connected colored regions in the colored figure\n    c_vis = np.zeros((c_h, c_w), dtype=bool)\n    color_regions = []\n    for r in range(c_h):\n        for c in range(c_w):\n            val = int(c_sub[r, c])\n            if val != bg and val != border_color and not c_vis[r, c]:\n                q = deque([(r, c)])\n                c_vis[r, c] = True\n                cells = []\n                while q:\n                    cr, cc = q.popleft()\n                    cells.append((cr, cc))\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < c_h and 0 <= nc < c_w and not c_vis[nr, nc] and c_sub[nr, nc] == val:\n                            c_vis[nr, nc] = True\n                            q.append((nr, nc))\n                mean_r = np.mean([pr for pr, _ in cells]) / max(1, c_h - 1)\n                mean_c = np.mean([pc for _, pc in cells]) / max(1, c_w - 1)\n                color_regions.append((mean_r, mean_c, val))\n\n    # 6. Match each wireframe hole to the nearest colored region and fill\n    for hole in holes:\n        hr = np.mean([pr for pr, _ in hole]) / max(1, w_h - 1)\n        hc = np.mean([pc for _, pc in hole]) / max(1, w_w - 1)\n        best_color = min(\n            color_regions,\n            key=lambda reg: (reg[0] - hr) ** 2 + (reg[1] - hc) ** 2\n        )[2]\n        for r, c in hole:\n            out[r, c] = best_color\n\n    return out.tolist()\n", "3dc255db.py": "# ARC-AGI-2 task 3dc255db: verified on all train pairs\n# model=gemini-4-argon sample=1 round=1 test_correct=1.0\n\nfrom collections import deque\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    W = len(grid[0])\n    out = [row[:] for row in grid]\n\n    visited = [[False] * W for _ in range(H)]\n    components = []\n\n    for r in range(H):\n        for c in range(W):\n            if grid[r][c] != 0 and not visited[r][c]:\n                color = grid[r][c]\n                q = deque([(r, c)])\n                visited[r][c] = True\n                cells = []\n                while q:\n                    cr, cc = q.popleft()\n                    cells.append((cr, cc))\n                    for dr in (-1, 0, 1):\n                        for dc in (-1, 0, 1):\n                            if dr == 0 and dc == 0:\n                                continue\n                            nr, nc = cr + dr, cc + dc\n                            if 0 <= nr < H and 0 <= nc < W and not visited[nr][nc] and grid[nr][nc] == color:\n                                visited[nr][nc] = True\n                                q.append((nr, nc))\n                rmin = min(cr for cr, _ in cells)\n                rmax = max(cr for cr, _ in cells)\n                cmin = min(cc for _, cc in cells)\n                cmax = max(cc for _, cc in cells)\n                components.append((color, set(cells), (rmin, rmax, cmin, cmax)))\n\n    for i, (color, cells, (rmin, rmax, cmin, cmax)) in enumerate(components):\n        if len(cells) < 5 or (rmax - rmin + 1) < 3 or (cmax - cmin + 1) < 3:\n            continue\n        # Ensure this component's bounding box is not strictly inside another component's bounding box\n        if any(\n            j != i and br0 <= rmin and rmax <= br1 and bc0 <= cmin and cmax <= bc1\n            and (br0, br1, bc0, bc1) != (rmin, rmax, cmin, cmax)\n            for j, (_, _, (br0, br1, bc0, bc1)) in enumerate(components)\n        ):\n            continue\n\n        inner = [\n            (r, c)\n            for r in range(rmin, rmax + 1)\n            for c in range(cmin, cmax + 1)\n            if grid[r][c] != 0 and grid[r][c] != color\n        ]\n        if not inner:\n            continue\n\n        dust_color = grid[inner[0][0]][inner[0][1]]\n        count = len(inner)\n        for r, c in inner:\n            out[r][c] = 0\n\n        # Identify the nozzle tip and emission direction\n        tip = None\n        direction = None\n\n        top_cells = [c for r, c in cells if r == rmin]\n        if len(top_cells) == 1 and cmin < top_cells[0] < cmax:\n            c0 = top_cells[0]\n            col_rows = sorted(r for r, c in cells if c == c0)\n            if col_rows == list(range(rmin, rmin + len(col_rows))):\n                tip = (rmin, c0)\n                direction = (-1, 0)\n\n        bottom_cells = [c for r, c in cells if r == rmax]\n        if tip is None and len(bottom_cells) == 1 and cmin < bottom_cells[0] < cmax:\n            c0 = bottom_cells[0]\n            col_rows = sorted(r for r, c in cells if c == c0)\n            if col_rows == list(range(rmax - len(col_rows) + 1, rmax + 1)):\n                tip = (rmax, c0)\n                direction = (1, 0)\n\n        left_cells = [r for r, c in cells if c == cmin]\n        if tip is None and len(left_cells) == 1 and rmin < left_cells[0] < rmax:\n            r0 = left_cells[0]\n            row_cols = sorted(c for r, c in cells if r == r0)\n            if row_cols == list(range(cmin, cmin + len(row_cols))):\n                tip = (r0, cmin)\n                direction = (0, -1)\n\n        right_cells = [r for r, c in cells if c == cmax]\n        if tip is None and len(right_cells) == 1 and rmin < right_cells[0] < rmax:\n            r0 = right_cells[0]\n            row_cols = sorted(c for r, c in cells if r == r0)\n            if row_cols == list(range(cmax - len(row_cols) + 1, cmax + 1)):\n                tip = (r0, cmax)\n                direction = (0, 1)\n\n        if tip is not None and direction is not None:\n            tr, tc = tip\n            dr, dc = direction\n            for k in range(1, count + 1):\n                nr, nc = tr + k * dr, tc + k * dc\n                if 0 <= nr < H and 0 <= nc < W:\n                    out[nr][nc] = dust_color\n\n    return out\n", "3e6067c3.py": "# ARC-AGI-2 task 3e6067c3: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H = len(grid)\n    W = len(grid[0])\n    out = [row[:] for row in grid]\n\n    # Background color is the most frequent color in the grid\n    counts = Counter(val for row in grid for val in row)\n    bg_color = counts.most_common(1)[0][0]\n\n    # Find connected components of non-background cells\n    visited = [[False] * W for _ in range(H)]\n    boxes = []\n    seq_pixels = []\n\n    for r in range(H):\n        for c in range(W):\n            if grid[r][c] != bg_color and not visited[r][c]:\n                stack = [(r, c)]\n                visited[r][c] = True\n                cells = []\n                while stack:\n                    cr, cc = stack.pop()\n                    cells.append((cr, cc))\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < H and 0 <= nc < W and not visited[nr][nc] and grid[nr][nc] != bg_color:\n                            visited[nr][nc] = True\n                            stack.append((nr, nc))\n\n                if len(cells) == 1:\n                    seq_pixels.append(cells[0])\n                else:\n                    r1 = min(cr for cr, _ in cells)\n                    r2 = max(cr for cr, _ in cells)\n                    c1 = min(cc for _, cc in cells)\n                    c2 = max(cc for _, cc in cells)\n                    frame_color = grid[r1][c1]\n                    inner_cells = [\n                        (cr, cc)\n                        for cr in range(r1, r2 + 1)\n                        for cc in range(c1, c2 + 1)\n                        if grid[cr][cc] != frame_color\n                    ]\n                    ir1 = min(cr for cr, _ in inner_cells)\n                    ir2 = max(cr for cr, _ in inner_cells)\n                    ic1 = min(cc for _, cc in inner_cells)\n                    ic2 = max(cc for _, cc in inner_cells)\n                    center_color = grid[ir1][ic1]\n                    boxes.append({\n                        \"r1\": r1, \"r2\": r2, \"c1\": c1, \"c2\": c2,\n                        \"ir1\": ir1, \"ir2\": ir2, \"ic1\": ic1, \"ic2\": ic2,\n                        \"color\": center_color,\n                    })\n\n    # Order the bottom sequence pixels from left to right\n    seq_pixels.sort(key=lambda pos: pos[1])\n    sequence = [grid[r][c] for r, c in seq_pixels]\n\n    # Build adjacency list between aligned boxes with no other box in between\n    n_boxes = len(boxes)\n    adj = [[] for _ in range(n_boxes)]\n    for i in range(n_boxes):\n        for j in range(i + 1, n_boxes):\n            A, B = boxes[i], boxes[j]\n            if A[\"r1\"] == B[\"r1\"] and A[\"r2\"] == B[\"r2\"]:\n                c_start = min(A[\"c2\"], B[\"c2\"]) + 1\n                c_end = max(A[\"c1\"], B[\"c1\"]) - 1\n                if all(grid[A[\"r1\"]][c] == bg_color for c in range(c_start, c_end + 1)):\n                    adj[i].append(j)\n                    adj[j].append(i)\n            elif A[\"c1\"] == B[\"c1\"] and A[\"c2\"] == B[\"c2\"]:\n                r_start = min(A[\"r2\"], B[\"r2\"]) + 1\n                r_end = max(A[\"r1\"], B[\"r1\"]) - 1\n                if all(grid[r][A[\"c1\"]] == bg_color for r in range(r_start, r_end + 1)):\n                    adj[i].append(j)\n                    adj[j].append(i)\n\n    # Find the path of boxes matching the color sequence\n    path = []\n    used = [False] * n_boxes\n\n    def dfs(idx: int) -> bool:\n        if idx == len(sequence):\n            return True\n        target_color = sequence[idx]\n        candidates = range(n_boxes) if idx == 0 else adj[path[-1]]\n        for nxt in candidates:\n            if not used[nxt] and boxes[nxt][\"color\"] == target_color:\n                used[nxt] = True\n                path.append(nxt)\n                if dfs(idx + 1):\n                    return True\n                path.pop()\n                used[nxt] = False\n        return False\n\n    dfs(0)\n\n    # Draw bridges between consecutive boxes in the path\n    for k in range(len(path) - 1):\n        A = boxes[path[k]]\n        B = boxes[path[k + 1]]\n        color = A[\"color\"]\n        if A[\"r1\"] == B[\"r1\"]:\n            c_start = min(A[\"c2\"], B[\"c2\"]) + 1\n            c_end = max(A[\"c1\"], B[\"c1\"]) - 1\n            for r in range(A[\"ir1\"], A[\"ir2\"] + 1):\n                for c in range(c_start, c_end + 1):\n                    out[r][c] = color\n        else:\n            r_start = min(A[\"r2\"], B[\"r2\"]) + 1\n            r_end = max(A[\"r1\"], B[\"r1\"]) - 1\n            for r in range(r_start, r_end + 1):\n                for c in range(A[\"ic1\"], A[\"ic2\"] + 1):\n                    out[r][c] = color\n\n    return out\n", "409aa875.py": "# ARC-AGI-2 task 409aa875: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    H, W = len(grid), len(grid[0])\n    out = [row[:] for row in grid]\n\n    # Determine background color (most frequent color)\n    bg = Counter(val for row in grid for val in row).most_common(1)[0][0]\n\n    # Find 8-connected components of non-background cells\n    visited = [[False] * W for _ in range(H)]\n    components = []\n    cell_to_comp = {}\n\n    for r in range(H):\n        for c in range(W):\n            if grid[r][c] != bg and not visited[r][c]:\n                comp = []\n                stack = [(r, c)]\n                visited[r][c] = True\n                while stack:\n                    cr, cc = stack.pop()\n                    comp.append((cr, cc))\n                    for nr in range(max(0, cr - 1), min(H, cr + 2)):\n                        for nc in range(max(0, cc - 1), min(W, cc + 2)):\n                            if not visited[nr][nc] and grid[nr][nc] != bg:\n                                visited[nr][nc] = True\n                                stack.append((nr, nc))\n                comp_idx = len(components)\n                components.append(comp)\n                for cell in comp:\n                    cell_to_comp[cell] = comp_idx\n\n    # Determine the vertex, direction, and target (5 steps away) for each arrow\n    hits = Counter()\n    for comp in components:\n        vertex = min(\n            comp,\n            key=lambda p: sum((p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2 for q in comp),\n        )\n        sum_dr = sum(vertex[0] - q[0] for q in comp)\n        sum_dc = sum(vertex[1] - q[1] for q in comp)\n        dr = (sum_dr > 0) - (sum_dr < 0)\n        dc = (sum_dc > 0) - (sum_dc < 0)\n\n        target = (vertex[0] + 5 * dr, vertex[1] + 5 * dc)\n        if 0 <= target[0] < H and 0 <= target[1] < W:\n            hits[target] += 1\n\n    # Apply target hits to the output grid\n    for (tr, tc), count in hits.items():\n        color = 9 if count == 1 else 1\n        if (tr, tc) in cell_to_comp:\n            for cr, cc in components[cell_to_comp[(tr, tc)]]:\n                out[cr][cc] = color\n        else:\n            out[tr][tc] = color\n\n    return out\n", "446ef5d2.py": "# ARC-AGI-2 task 446ef5d2: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nfrom collections import Counter, deque\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid, dtype=int)\n    R, C = arr.shape\n\n    # 1. Background colour is the most frequent colour\n    counts = Counter(arr.flatten())\n    bg = counts.most_common(1)[0][0]\n\n    # 2. Find connected components of identical colour to locate the 3-cell L-marker\n    visited = np.zeros((R, C), dtype=bool)\n    marker_cells = None\n    corner_cell = None\n    marker_box = None\n\n    for r in range(R - 1):\n        for c in range(C - 1):\n            window = [(r, c), (r, c + 1), (r + 1, c), (r + 1, c + 1)]\n            vals = [arr[pr, pc] for pr, pc in window]\n            if bg in vals:\n                continue\n            val_counts = Counter(vals)\n            if len(val_counts) == 2 and 3 in val_counts.values():\n                m_col = [k for k, v in val_counts.items() if v == 3][0]\n                if counts[m_col] == 3:\n                    marker_cells = [(pr, pc) for pr, pc in window if arr[pr, pc] == m_col]\n                    corner_cell = [(pr, pc) for pr, pc in window if arr[pr, pc] != m_col][0]\n                    marker_box = (r, c)\n                    break\n        if marker_cells is not None:\n            break\n\n    # Remove marker cells and extract all 4-connected non-background pieces\n    work = arr.copy()\n    for mr, mc in marker_cells:\n        work[mr, mc] = bg\n\n    pieces = []\n    fixed_idx = None\n    vis = np.zeros((R, C), dtype=bool)\n    for r in range(R):\n        for c in range(C):\n            if work[r, c] != bg and not vis[r, c]:\n                q = deque([(r, c)])\n                vis[r, c] = True\n                comp = []\n                while q:\n                    cr, cc = q.popleft()\n                    comp.append((cr, cc, int(work[cr, cc])))\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if 0 <= nr < R and 0 <= nc < C and not vis[nr, nc] and work[nr, nc] != bg:\n                            vis[nr, nc] = True\n                            q.append((nr, nc))\n                comp.sort()\n                if any((cr, cc) == corner_cell for cr, cc, _ in comp):\n                    fixed_idx = len(pieces)\n                pieces.append(comp)\n\n    total_area = sum(len(p) for p in pieces)\n    r_c, c_c = corner_cell\n    r_m, c_m = marker_box\n    border_color = int(work[r_c, c_c])\n\n    # Normalize each piece relative to its first cell in row-major order\n    norm_pieces = []\n    for p in pieces:\n        r0, c0, _ = p[0]\n        rel = [(r - r0, c - c0, val) for r, c, val in p]\n        rs = [r for r, _, _ in p]\n        cs = [c for _, c, _ in p]\n        h = max(rs) - min(rs) + 1\n        w = max(cs) - min(cs) + 1\n        norm_pieces.append((rel, h, w, p))\n\n    max_h = max(h for _, h, _, _ in norm_pieces)\n    max_w = max(w for _, _, w, _ in norm_pieces)\n\n    best_score = -1\n    best_board = None\n    best_rect_origin = None\n\n    for H in range(max_h, R + 1):\n        if total_area % H != 0:\n            continue\n        W = total_area // H\n        if W < max_w or W > C:\n            continue\n\n        # Determine top-left of the target HxW rectangle in the grid\n        r_top = r_c - H + 1 if r_c == r_m else r_c\n        c_left = c_c - W + 1 if c_c == c_m else c_c\n        if not (0 <= r_top and r_top + H <= R and 0 <= c_left and c_left + W <= C):\n            continue\n\n        board = np.full((H, W), -1, dtype=int)\n        owner = np.full((H, W), -1, dtype=int)\n\n        # Place the anchored piece at its original grid coordinates\n        possible = True\n        for pr, pc, val in pieces[fixed_idx]:\n            br, bc = pr - r_top, pc - c_left\n            if not (0 <= br < H and 0 <= bc < W):\n                possible = False\n                break\n            board[br, bc] = val\n            owner[br, bc] = fixed_idx\n        if not possible:\n            continue\n\n        rem_indices = [i for i in range(len(pieces)) if i != fixed_idx]\n        used = [False] * len(rem_indices)\n\n        def solve(first_idx: int):\n            nonlocal best_score, best_board, best_rect_origin\n            while first_idx < H * W and board[first_idx // W, first_idx % W] != -1:\n                first_idx += 1\n\n            if first_idx == H * W:\n                # Score the completed tiling by internal seam matches + uniform border matches\n                score = 0\n                for r in range(H):\n                    for c in range(W):\n                        if r in (0, H - 1) or c in (0, W - 1):\n                            if board[r, c] == border_color:\n                                score += 1\n                        if r + 1 < H and owner[r, c] != owner[r + 1, c]:\n                            if board[r, c] == board[r + 1, c]:\n                                score += 2\n                        if c + 1 < W and owner[r, c] != owner[r, c + 1]:\n                            if board[r, c] == board[r, c + 1]:\n                                score += 2\n                if score > best_score:\n                    best_score = score\n                    best_board = board.copy()\n                    best_rect_origin = (r_top, c_left)\n                return\n\n            r0, c0 = divmod(first_idx, W)\n            for k, p_idx in enumerate(rem_indices):\n                if used[k]:\n                    continue\n                rel, _, _, _ = norm_pieces[p_idx]\n                fits = True\n                for dr, dc, _ in rel:\n                    nr, nc = r0 + dr, c0 + dc\n                    if not (0 <= nr < H and 0 <= nc < W and board[nr, nc] == -1):\n                        fits = False\n                        break\n                if not fits:\n                    continue\n\n                used[k] = True\n                for dr, dc, val in rel:\n                    board[r0 + dr, c0 + dc] = val\n                    owner[r0 + dr, c0 + dc] = p_idx\n\n                solve(first_idx + 1)\n\n                for dr, dc, _ in rel:\n                    board[r0 + dr, c0 + dc] = -1\n                    owner[r0 + dr, c0 + dc] = -1\n                used[k] = False\n\n        solve(0)\n\n    out = np.full((R, C), bg, dtype=int)\n    r_top, c_left = best_rect_origin\n    H, W = best_board.shape\n    out[r_top : r_top + H, c_left : c_left + W] = best_board\n    return out.tolist()\n", "45a5af55.py": "# ARC-AGI-2 task 45a5af55: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    h = len(grid)\n    n = 2 * h - 2\n    return [\n        [grid[min(r, c, n - 1 - r, n - 1 - c)][0] for c in range(n)]\n        for r in range(n)\n    ]\n", "4a21e3da.py": "# ARC-AGI-2 task 4a21e3da: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    H, W = arr.shape\n    out = np.ones((H, W), dtype=int)\n    \n    # Locate laser sources on the four borders\n    top_c = None\n    bottom_c = None\n    left_r = None\n    right_r = None\n    \n    for c in range(W):\n        if arr[0, c] == 2:\n            top_c = c\n        if arr[H - 1, c] == 2:\n            bottom_c = c\n    for r in range(H):\n        if arr[r, 0] == 2:\n            left_r = r\n        if arr[r, W - 1] == 2:\n            right_r = r\n            \n    v_cut = top_c if top_c is not None else bottom_c\n    h_cut = left_r if left_r is not None else right_r\n    \n    # Draw vertical laser beam\n    if top_c is not None:\n        c = top_c\n        sevens = np.where(arr[:, c] == 7)[0]\n        max_r = sevens.max() if len(sevens) > 0 else 0\n        for r in range(0, max_r + 1):\n            out[r, c] = 7 if arr[r, c] == 7 else 2\n            \n    if bottom_c is not None:\n        c = bottom_c\n        sevens = np.where(arr[:, c] == 7)[0]\n        min_r = sevens.min() if len(sevens) > 0 else H - 1\n        for r in range(min_r, H):\n            out[r, c] = 7 if arr[r, c] == 7 else 2\n            \n    # Draw horizontal laser beam\n    if left_r is not None:\n        r = left_r\n        sevens = np.where(arr[r, :] == 7)[0]\n        max_c = sevens.max() if len(sevens) > 0 else 0\n        for c in range(0, max_c + 1):\n            out[r, c] = 7 if arr[r, c] == 7 else 2\n            \n    if right_r is not None:\n        r = right_r\n        sevens = np.where(arr[r, :] == 7)[0]\n        min_c = sevens.min() if len(sevens) > 0 else W - 1\n        for c in range(min_c, W):\n            out[r, c] = 7 if arr[r, c] == 7 else 2\n\n    # Helper to shift a piece's bounding box into a corner\n    def place_in_corner(mask, align_bottom, align_right):\n        rs, cs = np.where(mask & (arr == 7))\n        if len(rs) == 0:\n            return\n        r_shift = (H - 1 - rs.max()) if align_bottom else -rs.min()\n        c_shift = (W - 1 - cs.max()) if align_right else -cs.min()\n        for r, c in zip(rs, cs):\n            out[r + r_shift, c + c_shift] = 7\n\n    R_idx, C_idx = np.indices((H, W))\n\n    if v_cut is not None and h_cut is not None:\n        if top_c is not None or left_r is not None:\n            place_in_corner((R_idx < h_cut) & (C_idx < v_cut), align_bottom=False, align_right=False)\n        if top_c is not None or right_r is not None:\n            place_in_corner((R_idx < h_cut) & (C_idx > v_cut), align_bottom=False, align_right=True)\n        if bottom_c is not None or left_r is not None:\n            place_in_corner((R_idx > h_cut) & (C_idx < v_cut), align_bottom=True, align_right=False)\n        if bottom_c is not None or right_r is not None:\n            place_in_corner((R_idx > h_cut) & (C_idx > v_cut), align_bottom=True, align_right=True)\n    elif v_cut is not None:\n        align_bottom = (bottom_c is not None)\n        place_in_corner(C_idx < v_cut, align_bottom=align_bottom, align_right=False)\n        place_in_corner(C_idx > v_cut, align_bottom=align_bottom, align_right=True)\n    elif h_cut is not None:\n        align_right = (right_r is not None)\n        place_in_corner(R_idx < h_cut, align_bottom=False, align_right=align_right)\n        place_in_corner(R_idx > h_cut, align_bottom=True, align_right=align_right)\n\n    return out.tolist()\n", "4c3d4a41.py": "# ARC-AGI-2 task 4c3d4a41: verified on all train pairs\n# model=gemini-4-argon sample=1 round=0 test_correct=1.0\n\nimport numpy as np\n\n\ndef transform(grid: list[list[int]]) -> list[list[int]]:\n    arr = np.array(grid)\n    out = arr.copy()\n    h, w = arr.shape\n\n    # Locate the lock (the frame that touches the top row) and the key on the left\n    lock_cols = np.where(arr[0] == 5)[0]\n    c_lock = int(lock_cols[0])\n\n    key_mask = np.zeros_like(arr, dtype=bool)\n    key_mask[:, :c_lock] = arr[:, :c_lock] == 5\n    key_rows, key_cols = np.where(key_mask)\n\n    c_key_start = int(key_cols.min())\n    c_key_end = int(key_cols.max())\n    r_base = int(key_rows.max())\n\n    # Remove the key from its initial position\n    out[key_mask] = 0\n\n    # Horizontal shift aligns the first key prong with the first pin column inside the lock\n    shift = (c_lock + 2) - c_key_start\n\n    for c_k in range(c_key_start, c_key_end + 1):\n        c_p = c_k + shift\n        prong_rows = np.where(arr[:, c_k] == 5)[0]\n        if len(prong_rows) == 0:\n            continue\n        t = int(prong_rows.min())\n\n        # Check for an existing pin in the target lock column (rows 1 .. r_base)\n        pin_rows = np.where(arr[1 : r_base + 1, c_p] != 0)[0] + 1\n        if len(pin_rows) > 0:\n            b = int(pin_rows.max())\n            up_shift = max(0, b - t + 1)\n            for r in range(1, t):\n                src_r = r + up_shift\n                out[r, c_p] = arr[src_r, c_p] if src_r <= b else 0\n\n        # Place the key column in the lock\n        out[t : r_base + 1, c_p] = 5\n\n    return out.tolist()\n"}
os.makedirs('/kaggle/working/arc/library', exist_ok=True)
open('/kaggle/working/arc/pipeline.py', 'w').write(PIPELINE_SRC)
for name, src in PROGRAMS.items():
    open(os.path.join('/kaggle/working/arc/library', name), 'w').write(src)
print(len(PROGRAMS), 'library programs written')

In [ ]:
import glob
cands = sorted(glob.glob('/kaggle/input/**/*test_challenges*.json', recursive=True))
assert cands, 'test challenges file not found under /kaggle/input'
CHALLENGES = cands[0]
print('Using', CHALLENGES)

In [ ]:
!python /kaggle/working/arc/pipeline.py --challenges "$CHALLENGES" --out-dir /tmp/arc_out \
    --time-per-task 8 --program-library /kaggle/working/arc/library

In [ ]:
import shutil
shutil.copy('/tmp/arc_out/submission.json', '/kaggle/working/submission.json')
sub = json.load(open('/kaggle/working/submission.json'))
print('submission.json written for', len(sub), 'tasks')